In [0]:
# Databricks notebook source
# MAGIC %pip install sparkmeasure
# MAGIC %pip install azure-storage-blob
# MAGIC # %pip install protobuf==3.17.2
# MAGIC %pip install /dbfs/FileStore/TigerML_045/tigerml.core-0.4.5-py3-none-any.whl --no-deps
# MAGIC %pip install holoviews==1.14.9
# MAGIC %pip install python-slugify==3.0.4

# COMMAND ----------

from sparkmeasure import StageMetrics
from sparkmeasure import TaskMetrics

taskmetrics = TaskMetrics(spark)
stagemetrics = StageMetrics(spark)

taskmetrics.begin()
stagemetrics.begin()

# COMMAND ----------

# MAGIC %md
# MAGIC ##Imports

# COMMAND ----------

import os
import ast
import requests
from requests.structures import CaseInsensitiveDict
import time
import requests
from datetime import datetime
from delta.tables import *
import pandas as pd
import numpy as np
import pyspark
from pyspark.sql import types as DT, functions as F, Window
import ast
import traceback
import sys
import json
import pandas as pd
from tenacity import retry, stop_after_delay, stop_after_attempt, wait_exponential
import re
import copy
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    DateType,
    IntegerType,
    LongType,
)
from pyspark.sql import types as T
from pyspark.sql.types import *

from monitoring.utils.secret_mapping import SECRET_MAPPING

# DONT SHOW SETTINGWITHCOPY WARNING
pd.options.mode.chained_assignment = None
from monitoring.utils import utils

# Importing TSL Adaptor scripts
from Native_Adaptors.adaptor import get_monitoring_tables

# COMMAND ----------

# MAGIC %md
# MAGIC ## Utility Functions

# COMMAND ----------

import yaml
try:
    solution_config = (dbutils.widgets.get("solution_config"))
    solution_config = yaml.safe_load(solution_config)
except:
    with open('../data_config/SolutionConfig.yaml', 'r') as solution_config:
        solution_config = yaml.safe_load(solution_config)

try :
    env = dbutils.widgets.get("env")
except :
    env = "dev"
print(f"Input environment : {env}")

try:
    sdk_session_id = dbutils.widgets.get("sdk_session_id")
except:
    sdk_session_id = solution_config.get("general_configs").get("sdk_session_id").get(env)
    # sdk_session_id = solution_config[f'sdk_session_id_{env}']

if sdk_session_id.lower() == "none":
    sdk_session_id = solution_config[f'sdk_session_id_{env}']

# COMMAND ----------


def get_env_vault_scope():
    """
    Returns env and vault scope
    """
    import json

    env = (
        dbutils.notebook.entry_point.getDbutils()
        .notebook()
        .getContext()
        .notebookPath()
        .get()
    ).split("/")[2]
    try:
        if len(dbutils.fs.ls("dbfs:/FileStore/jars/MLCORE_INIT/vault_check.json")) == 1:
            # if env == "qa":
            #     with open("/dbfs/FileStore/jars/MLCORE_INIT/vault_check_qa.json", "r") as file:
            #         vault_check_data = json.loads(file.read())
            # else:
            with open("/dbfs/FileStore/jars/MLCORE_INIT/vault_check.json", "r") as file:
                vault_check_data = json.loads(file.read())
            if "@" in env:
                return "qa", vault_check_data["client_name"]
            return env, vault_check_data["client_name"]
        else:
            return env, vault_scope
    except:
        return env, vault_scope


def get_access_tokens(client_id, scope, client_secret, vault_scope):
    """
    Returns a bearer token
    """

    headers = CaseInsensitiveDict()
    headers["Content-Type"] = "application/x-www-form-urlencoded"
    data = {}
    data["client_id"] = client_id
    data["grant_type"] = "client_credentials"
    data["scope"] = scope
    data["client_secret"] = client_secret
    tenant_id = dbutils.secrets.get(
        scope=vault_scope, key=SECRET_MAPPING.get("az-directory-tenant", "")
    )
    url = "https://login.microsoftonline.com/" + tenant_id + "/oauth2/v2.0/token"
    resp = requests.post(url, headers=headers, data=data).json()
    token = resp["access_token"]
    token_string = "Bearer" + " " + token
    return token_string


def get_app_url():
    """
    Returns env and vault scope
    """
    import json

    env = (
        dbutils.notebook.entry_point.getDbutils()
        .notebook()
        .getContext()
        .notebookPath()
        .get()
    ).split("/")[2]

    if env == "dev":
        api_endpoint = "https://mlcoredevv2pg21.azurewebsites.net/"
    elif env == "qa":
        api_endpoint = "https://mlcoredevv2pg21test.azurewebsites.net/"
    else:
        api_endpoint = "https://mlcoreuatv2pg21.azurewebsites.net/"

    return api_endpoint


def get_headers(vault_scope):
    """
    Returns API headers
    """
    h1 = CaseInsensitiveDict()
    client_id = dbutils.secrets.get(scope=vault_scope, key=SECRET_MAPPING.get("az-api-client-id",""))
    scope = client_id + "/.default"
    client_secret = dbutils.secrets.get(scope=vault_scope, key=SECRET_MAPPING.get("az-api-client-secret",""))
    h1["Authorization"] = get_access_tokens(
        client_id, scope, client_secret, vault_scope
    )
    h1["Content-Type"] = "application/json"
    return h1


def get_cluster_info(spark):
    p = "spark.databricks.clusterUsageTags."
    conf = spark.sparkContext.getConf().getAll()
    conf_dict = {k.replace(p, ""): v for k, v in conf if k.startswith(p)}

    return conf_dict


def get_params(name):
    import json

    param = dbutils.widgets.get(name)
    param = param.replace("'", '"')
    param = json.loads(param)
    return param


def fetch_id(transform=None):
    import json

    # Checking if task parameter variable are set
    task_info = dbutils.notebook.entry_point.getCurrentBindings()
    job_id = task_info.get("job_id", None)
    run_id = task_info.get("parent_run_id", None)

    # Getting notebook context to get task id and source information
    notebook_info = json.loads(
        dbutils.notebook.entry_point.getDbutils().notebook().getContext().toJson()
    )

    # If task parameter values are not extracted successfully, extracting job id and run id
    # from the notebook context
    if job_id == None or run_id == None:
        run_id = notebook_info["tags"]["multitaskParentRunId"]
        job_id = notebook_info["tags"]["jobId"]

    taskKey = "monitoring"
    taskDependency = ""
    if transform:
        taskDependencies = notebook_info["tags"]["taskDependencies"]
        taskDependenciesList = [x for x in taskDependencies.split('"') if "task" in x]
        if len(taskDependenciesList) == 1:
            return run_id, job_id, taskKey, taskDependenciesList[0]
        else:
            return run_id, job_id, taskKey, taskDependenciesList
    else:
        taskDependency = ""
        return run_id, job_id, taskKey, taskDependency


def get_monitoring_config(deployment_master_id):
    """
    Returns model monitoring configs as per the deployment master id
    """
    url = os.path.join(API_ENDPOINT, "mlapi/monitoring_configs/get")
    url = url + f"?deployment_master_id={deployment_master_id}"
    payload = {}
    response = requests.get(url=url, headers=h1, data=payload).json()
    utils.log(
        f"\n\
    endpoint - mlapi/monitoring_configs/get\n\
    response - {response}",
        message_run,
    )
    return response["data"]


def write_metrics_to_delta(metric_json, kind):
    """
    Takes in metric json and writes delta at a dedicated metrics delta path for the resp kind of metric list
    """
    try:
        # Determining target table path
        if kind != "performance_drift":
            if datalake_env.lower() == "delta":
                # metric_table_path = f"dbfs:/mnt/{az_container_name}/{env}/monitoring/{job_id}/data_model_drift"
                metric_table_path = f"dbfs:/user/hive/warehouse/{sdk_session_id}.db/data_model_drift_{job_id}"
            else:
                metric_table_path = f"{env}_monitoring_{job_id}_data_model_drift"
        else:
            if datalake_env.lower() == "delta":
                # metric_table_path = f"dbfs:/mnt/{az_container_name}/{env}/monitoring/{job_id}/performance_drift"
                metric_table_path = f"dbfs:/user/hive/warehouse/{sdk_session_id}.db/performance_drift_{job_id}"
            else:
                metric_table_path = f"{env}_monitoring_{job_id}_performance_drift"
        # DEL BELOW
        print(f"METRIC_JSON : {metric_json}")
        utils.log(f"METRIC_TABLE_PATH : {metric_table_path}", message_run)
        # Fetching target table schema
        schema = get_metric_table_schema(kind)

        # Empty string treatment for delta compatibility
        metric_json = json_encode_nans(metric_json)
        metric_json = delta_encode_str(metric_json)
        restructured_json = restructure_metric_json(metric_json, kind)

        colnames = [
            k
            for k, v in restructured_json[0].items()
            if k not in ["data", "monitoring_algo"]
        ]
        # DEL BELOW
        print(f"RESTRUCTURED_JSON - {restructured_json}")
        utils.log(f"COLNAMES - {colnames}", message_run)
        # Converting metric json into a dataframe
        pd_metric_df = pd.json_normalize(restructured_json, "data", colnames)

        # Ordering columns of the pandas daraframe as per the target table schema
        pd_metric_df = pd_metric_df[[entry.name for entry in schema.fields]]

        # If null exist in is_drift, pandas converts this int column into float, which breaks delta
        # Checking if this scenario is present, and typecasting to handle the scenario accordingly
        if (
            pd_metric_df.dtypes["is_drift"] != "int"
            and pd_metric_df["is_drift"].isna().sum() > 0
        ):
            pd_metric_df["is_drift"] = (
                pd_metric_df["is_drift"].astype("Int64").fillna(-1)
            )

        # Converting pandas dataframe into a pyspark dataframe
        metric_df = spark.createDataFrame(pd_metric_df, schema=schema)

        # Adding the date literal column
        metric_df = metric_df.withColumn("date", F.lit(date))
        metric_df = metric_df.withColumn("date", F.to_date(F.col("date"), "MM-dd-yyyy"))
        metric_df = metric_df.withColumn(
            "timestamp",
            F.expr("reflect('java.lang.System', 'currentTimeMillis')").cast("long"),
        )

        if kind != "performance_drift":
            metric_df = metric_df.withColumn(
                "modelling_task_name", F.lit(modelling_task_type)
            )

        print("METRIC_DF : ", message_run)
        metric_df.display()

        # Saving the table into the target path
        if table_already_created(metric_table_path):
            # Check the existing metrics data and take action based on below condition:
            # If "id" column is present, start the id from max_id + 1
            # If "id" column is not present, merge existing and new df and then restart the id from 1
            # If the table does not exists, simply start the id from 1
            existing_metric_table_data = read_data(metric_table_path)
            if "id" in existing_metric_table_data.columns:
                max_id = existing_metric_table_data.select(F.max("id")).collect()[0][0]
                w = Window.orderBy(F.lit(0))
                metric_df = metric_df.withColumn("id", F.row_number().over(w) + max_id)
            else:
                metric_df = existing_metric_table_data.union(metric_df)
                w = Window.orderBy(F.monotonically_increasing_id())
                metric_df = metric_df.withColumn("id", F.row_number().over(w))
            try:
                write_data(
                    data_path=metric_table_path,
                    dataframe=metric_df,
                    mode="append",
                    partition_by="date",
                )
            except:
                write_data(
                    data_path=metric_table_path,
                    dataframe=metric_df,
                    mode="overwrite",
                    partition_by="date",
                )
        else:
            w = Window.orderBy(F.monotonically_increasing_id())
            metric_df = metric_df.withColumn("id", F.row_number().over(w))
            write_data(
                data_path=metric_table_path,
                dataframe=metric_df,
                mode="append",
                partition_by="date",
            )

        # Write Aggregate Data to the table
        if kind != "performance_drift":
            mode = (
                "overwrite"
                if not table_already_created(aggregated_dd_drift_path, True)
                else "append"
            )
            write_data(
                data_path=aggregated_dd_drift_path,
                dataframe=metric_df,
                mode=mode,
                partition_by=[
                    "project_id",
                    "version",
                    "deployment_master_id",
                    "monitoring_algo_id",
                ],
                is_platform_table=True,
            )
        else:
            mode = (
                "overwrite"
                if not table_already_created(aggregated_performance_drift_path, True)
                else "append"
            )
            write_data(
                data_path=aggregated_performance_drift_path,
                dataframe=metric_df,
                mode=mode,
                partition_by=["project_id", "version", "deployment_master_id"],
                is_platform_table=True,
            )

        # Register table to Hive with the datalake env is delta
        if datalake_env.lower() == "delta":
            register_delta_as_hive(
                db_name, f"{kind}_{job_id}", metric_table_path, spark
            )

    except Exception as e:
        traceback.print_exc()
        # FIXME:
        # Commenting out this as it is causing inference_metadata to not be written
        # on monitorkits end
        # throw_exception(e)


def replace_empty_string_with_none(local_feat_entry):
    """
    If value of a key is an empty string, replaces it with None for delta datatype compatibility
    """
    for local_feat_entry_k, local_feat_entry_v in local_feat_entry.items():
        if local_feat_entry_v == "":
            local_feat_entry[local_feat_entry_k] = None
    return local_feat_entry


def restructure_metric_json(metric_json, kind):
    """
    Restructures generated metric json into delta template format
    """
    copy_metric_json = copy.deepcopy(metric_json)
    for entry in copy_metric_json:
        # restructing data object
        data = entry.get("data", {})
        restructured_data = []
        if kind == "feature_drift":
            for feat_entry, feat_entry_obj in data.items():
                local_feat_entry = {}
                if isinstance(feat_entry_obj, dict):
                    local_feat_entry["feature_attribute"] = (
                        feat_entry
                        if entry.get("monitoring_sub_type", "") == "feature_level"
                        else "overall"
                    )
                    local_feat_entry["is_drift"] = feat_entry_obj.get("is_drift", None)
                    local_feat_entry["stat_val"] = feat_entry_obj.get("stat_val", None)
                    local_feat_entry["p_val"] = feat_entry_obj.get("p_val", None)
                    local_feat_entry = replace_empty_string_with_none(local_feat_entry)
                    restructured_data.append(local_feat_entry)

        elif kind in ["target_drift", "concept_drift"]:
            local_feat_entry = {}
            local_feat_entry["feature_attribute"] = "overall"
            local_feat_entry["is_drift"] = data.get("is_drift", None)
            local_feat_entry["stat_val"] = data.get("stat_val", None)
            local_feat_entry["p_val"] = data.get("p_val", None)
            local_feat_entry = replace_empty_string_with_none(local_feat_entry)
            restructured_data.append(local_feat_entry)

        else:
            for feat_entry, feat_entry_obj in data.items():
                local_feat_entry = {}
                if isinstance(feat_entry_obj, dict):
                    local_feat_entry["feature_attribute"] = feat_entry
                    local_feat_entry["is_drift"] = feat_entry_obj.get("is_drift", None)
                    local_feat_entry["value"] = feat_entry_obj.get("value", None)
                else:
                    local_feat_entry["feature_attribute"] = "overall"
                    local_feat_entry["is_drift"] = data.get("is_drift", None)
                    local_feat_entry["value"] = data.get("value", None)
                local_feat_entry = replace_empty_string_with_none(local_feat_entry)
                restructured_data.append(local_feat_entry)

        if "feature_attribute" not in data.keys():
            data["feature_attribute"] = "overall"

        entry["data"] = restructured_data if len(restructured_data) != 0 else [data]

        # restructing monitoring algo
        moni_algo = entry.get("monitoring_algo", {})
        if kind != "performance_drift":
            entry["monitoring_algo_id"] = moni_algo.get("algo_id")
        entry["modelling_task_name"] = moni_algo.get("modelling_task_name")
        entry["monitoring_algo_name"] = moni_algo.get("algo_name")
        try:
            del entry["monitoring_algo"]
        except:
            pass
    return copy_metric_json


def check_if_upstream_job_active(job_id):
    """
    Checks if the job is active
    """
    try:
        if isinstance(job_id, list):
            job_id = ",".join(job_id)

        url = os.path.join(API_ENDPOINT, JOB_LOGS)
        url = url + f"?job_id={job_id}"
        headers = {"Authorization": h1["Authorization"]}
        response = requests.request("GET", url, headers=headers, data={})

        response = response.json()
        data = response.get("data", [])
        if not data:
            data = []

        status_of_jobs = [entry.get("status", "active") for entry in data]
        if "archived" not in status_of_jobs:
            return True
        else:
            return False

    except Exception as e:
        utils.log(str(e), message_run)
        return True


def declare_job_as_successful(
    no_data=False,
    no_monitor_config=False,
    no_inference_data=False,
    upstream_inactive=False,
    skipped_due_to_gt=[],
    drifted_monitor_types=[],
    aggregated_is_drift=False,
    processed_records="yes",
):
    """
    Setting the Monitor job status as successful by updating Job Runs Log and Job Tasks Log
    """
    stagemetrics.end()
    taskmetrics.end()

    stage_Df = stagemetrics.create_stagemetrics_DF("PerfStageMetrics")
    task_Df = taskmetrics.create_taskmetrics_DF("PerfTaskMetrics")

    aggregate_compute_metrics = (
        stagemetrics.aggregate_stagemetrics_DF()
        .select("executorCpuTime", "peakExecutionMemory")
        .collect()[0]
        .asDict()
    )

    aggregate_compute_metrics["executorCpuTime"] = (
        aggregate_compute_metrics["executorCpuTime"] / 1000
        if aggregate_compute_metrics["executorCpuTime"]
        else 0
    )
    aggregate_compute_metrics["peakExecutionMemory"] = (
        aggregate_compute_metrics["peakExecutionMemory"] / (1024 * 1024)
        if aggregate_compute_metrics["peakExecutionMemory"]
        else 0
    )

    compute_metrics = {
        "stagemetrics": stage_Df.rdd.map(lambda row: row.asDict()).collect(),
        "taskmetrics": task_Df.rdd.map(lambda row: row.asDict()).collect(),
    }

    end_time = str(int(time.time() * 1000000))
    utils.log("final logs into collection", message_run)
    run_id, job_id, table, source = fetch_id()
    model_inference_info = dbutils.widgets.get("model_inference_info")
    if isinstance(model_inference_info, str):
        model_inference_info = json.loads(model_inference_info)
    project_id, version = (
        model_inference_info["project_id"],
        model_inference_info["version"],
    )
    created_by_id, created_by_name = (
        model_inference_info["created_by_id"],
        model_inference_info["created_by_name"],
    )

    # Converting aggregated is drift flag from boolean to string for API compatibility
    if aggregated_is_drift:
        aggregated_is_drift = "yes"
    else:
        aggregated_is_drift = "no"

    # Fetching headers
    h1 = get_headers(vault_scope)

    if no_data:
        message_run.append(
            {
                "time": str(end_time),
                "message": "Monitor Job with job id: "
                + str(job_id)
                + " and run id: "
                + str(run_id)
                + " had no data to monitor.",
            }
        )
        message_task.append(
            {
                "time": str(end_time),
                "message": table
                + " within job id: "
                + str(job_id)
                + " and run id: "
                + str(run_id)
                + " had no data to monitor.",
            }
        )
    elif no_monitor_config:
        message_run.append(
            {
                "time": str(end_time),
                "message": "Monitor Job with job id: "
                + str(job_id)
                + " and run id: "
                + str(run_id)
                + " did not find monitoring configuration.",
            }
        )
        message_task.append(
            {
                "time": str(end_time),
                "message": table
                + " within job id: "
                + str(job_id)
                + " and run id: "
                + str(run_id)
                + " did not find monitoring configuration.",
            }
        )

    elif no_inference_data:
        message_run.append(
            {
                "time": str(end_time),
                "message": " Monitor Job with  job id: "
                + str(job_id)
                + " and run id: "
                + str(run_id)
                + " cannot proceed before at least one successful run of model inference gets completed.",
            }
        )
        message_task.append(
            {
                "time": str(end_time),
                "message": table
                + " within  job id: "
                + str(job_id)
                + " and run id: "
                + str(run_id)
                + " cannot proceed before at least one successful run of model inference gets completed.",
            }
        )
    elif upstream_inactive:
        message_run.append(
            {
                "time": str(end_time),
                "message": f"The data prep deployment pipeline or model inference pipeline, \
                            on which the model inference job with job id: {job_id} and run id: {run_id} has been triggered, is inactive.",
            }
        )
        message_task.append(
            {
                "time": str(end_time),
                "message": f"The data prep deployment pipeline or model inference pipeline, \
                            on which the model inference job with job id: {job_id} and run id: {run_id} has been triggered, is inactive.",
            }
        )
    elif len(null_gt_skipped_types) > 0:
        all_skipped_types = ", ".join(null_gt_skipped_types)
        message = (
            f"Monitor Job with job id: {job_id} and run id: {run_id} could not process {all_skipped_types} as there are null values in the ground truth "
            "for some of the features."
        )
        message_run.append(
            {
                "time": str(end_time),
                "message": message,
            }
        )
        message_task.append(
            {
                "time": str(end_time),
                "message": message,
            }
        )
    else:
        message_run.append(
            {
                "time": str(end_time),
                "message": "Monitor Job with job id: "
                + str(job_id)
                + " and run id: "
                + str(run_id)
                + " is finished successfully.",
            }
        )
        message_task.append(
            {
                "time": str(end_time),
                "message": table
                + " within job id: "
                + str(job_id)
                + " and run id: "
                + str(run_id)
                + " success.",
            }
        )

    log_data = {
        "project_id": project_id,
        "version": str(version),
        "job_id": str(job_id),
        "run_id": str(run_id),
        "end_time": str(end_time),
        "status": "success",
        "message": message_run,
        "updated_by_id": created_by_id,
        "updated_by_name": created_by_name,
        "created_by_id": created_by_id,
        "created_by_name": created_by_name,
        "task_id": "task0",
        "job_type": "Monitor",
        "drift_monitor_type": drifted_monitor_types,
        "is_drift": aggregated_is_drift,
        "processed_records": processed_records,
        "cpu": str(aggregate_compute_metrics.get("executorCpuTime", "NA")),
        "ram": str(aggregate_compute_metrics.get("peakExecutionMemory", "NA")),
        "cluster_info": get_cluster_info(spark),
        "compute_metrics": compute_metrics,
        "run_notebook_url": run_notebook_url,
        "deployment_master_id": deployment_master_id,
    }
    response = requests.put(API_ENDPOINT + JOB_RUNS_UPDATE, json=log_data, headers=h1)
    print(
        f"\n\
    Logging task:\n\
    endpoint - {JOB_RUNS_UPDATE}\n\
    status   - {response}\n\
    response - {response.text}\n\
    payload  - {log_data}\n"
    )

    utils.save_sparkmeasure_aggregated_tables(
        project_name=project_name,
        job_run_update_payload=log_data,
        compute_usage_metrics=aggregate_compute_metrics,
        stagemetrics=stagemetrics,
        taskmetrics=taskmetrics,
        api_endpoint=API_ENDPOINT,
        headers=h1,
        spark=spark,
        dbutils=dbutils,
    )

    task_log_data = {
        "job_id": str(job_id),
        "run_id": str(run_id),
        "task_id": "task0",
        "end_time": str(end_time),
        "status": "success",
        "message": message_task,
        "updated_by_id": created_by_id,
        "updated_by_name": created_by_name,
        "created_by_id": created_by_id,
        "created_by_name": created_by_name,
        "job_type": "Monitor",
        "run_notebook_url": run_notebook_url,
    }
    response = requests.put(
        API_ENDPOINT + JOB_TASK_UPDATE, json=task_log_data, headers=h1
    )
    print(
        f"\n\
    Logging task:\n\
    endpoint - {JOB_TASK_UPDATE}\n\
    status   - {response}\n\
    response - {response.text}\n\
    payload  - {task_log_data}\n"
    )

    # Exiting the notebook
    try:
        dbutils.notebook.exit("Job is Successful!")
    except:
        pass


def generate_run_notebook_url(job_id, run_id):
    """
    Generates the databricks job run notebook url in runtime
    """
    workspace_url = "https://" + spark.conf.get("spark.databricks.workspaceUrl")
    workspace_id = spark.conf.get("spark.databricks.clusterUsageTags.clusterOwnerOrgId")
    run_notebook_url = f"{workspace_url}/?o={workspace_id}#job/{job_id}/run/{run_id}"
    return run_notebook_url


@retry(
    wait=wait_exponential(min=4, multiplier=1, max=10),
    stop=(stop_after_delay(10) | stop_after_attempt(5)),
)
def table_exists(
    table_type, table_sub_type, job_id, version, project_id, is_platform=False
):
    """
    Returns if there is a table present for the given job_id and task_id combination
    """

    params = {
        "type": table_type,
        "sub_type": table_sub_type,
        "job_id": job_id,
        "project_id": project_id,
        "version": version,
        "mode": "detail",
    }
    headers = get_headers(vault_scope)
    response = requests.get(API_ENDPOINT + LIST_TABLES, params=params, headers=h1)
    utils.log(
        f"\n\
    endpoint - {LIST_TABLES}\n\
    payload  - {params}\n\
    response - {response}",
        message_run,
    )

    if response.status_code not in [200, 201]:
        raise Exception(
            f"API Error : The {LIST_TABLES} API returned {response.status_code} status code."
        )
    response = response.json()
    if response["data"]:
        if is_platform:
            for data in response["data"]:
                if data["datalake_env"] == platform_datalake_env:
                    return data
                else:
                    return None
        else:
            return response
    else:
        return None


def tables_add(data):
    """
    Calls tables add API with the entered data
    """
    h1 = get_headers(vault_scope)
    response = requests.post(API_ENDPOINT + TABLES_ADD, json=data, headers=h1)
    utils.log(
        f"\n\
    endpoint - {TABLES_ADD}\n\
    payload  - {data}\n\
    response - {response}",
        message_run,
    )

    # Raise exception if table_add is failing
    if response.status_code not in [200, 201]:
        try:
            response_message = response.json()
        except:
            response_message = ""

        raise Exception(
            f"API Error : The {TABLES_ADD} API returned {response.status_code} status code. Response : {response_message}"
        )
    return response


def get_table_dbfs_path(table_type, table_sub_type, table_id=None):
    """
    Returns templated dbfs path as per the table type and sub type
    """
    if table_type == "Monitoring_Output":
        if table_sub_type == "Data_Model":
            if datalake_env.lower() == "delta":
                path = f"dbfs:/mnt/{az_container_name}/{env}/monitoring/{job_id}/data_model_drift"
            else:
                path = f"{env}_monitoring_{job_id}_data_model_drift"
        else:
            if datalake_env.lower() == "delta":
                path = f"dbfs:/mnt/{az_container_name}/{env}/monitoring/{job_id}/performance_drift"
            else:
                path = f"{env}_monitoring_{job_id}_performance_drift"

    elif table_type == "Task_Log":
        path = f"dbfs:/mnt/{az_container_name}/{env}/{project_id}/{version}/{job_id}/task_log_table"
    if table_id:
        h1 = get_headers(vault_scope)
        params={"table_id": table_id}
        response = requests.get(
            API_ENDPOINT + TABLES_METADATA, params=params, headers=h1
        )
        print(response)
        utils.log(
            f"\n\
        endpoint - {TABLES_ADD}\n\
        payload  - {params}\n\
        response - {response}",
            message_run,
        )
        print(response.json())
        path = response.json()["data"][0]["dbfs_path"]
    return path


@retry(
    wait=wait_exponential(min=4, multiplier=1, max=10),
    stop=(stop_after_delay(40) | stop_after_attempt(5)),
)
def add_table_in_mongo(table_type, table_sub_type):
    """
    Adds table in mongo as per the table type and sub type
    """
    # Fetching templated dbfs path
    table_path = get_table_dbfs_path(table_type, table_sub_type)
    if table_type == "Monitoring_Output" and ("data_model_segmented" in table_sub_type.lower()):
        table_path = f"dbfs:/user/hive/warehouse/{sdk_session_id}.db/data_model_drift_{job_id}"
    if table_type == "Monitoring_Output" and ("performance_segmented" in table_sub_type.lower()):
        table_path = f"dbfs:/user/hive/warehouse/{sdk_session_id}.db/performance_drift_{job_id}"
    # local primary keys
    if table_type == "Monitoring_Output":
        local_primary_keys = [
            "deployment_master_id",
            "monitoring_type",
            "monitoring_sub_type",
            "job_id",
            "run_id",
        ]
    else:
        local_primary_keys = ["monitoring_subtype", "job_id", "run_id"]

    # Defining payload
    data = {
        "name": (
            f"{table_sub_type}_drift"
            if table_type == "Monitoring_Output"
            else "Task_Log_Table"
        ),
        "type": table_type,
        "sub_type": table_sub_type,
        "job_id": str(job_id),
        "created_run_id": str(run_id),
        "project_id": str(project_id),
        "version": str(version),
        "created_by_id": str(created_by_id),
        "created_by_name": created_by_name,
        "updated_by_id": str(created_by_id),
        "updated_by_name": created_by_name,
        "deployment_master_id": str(deployment_master_id),
        "primary_keys": local_primary_keys,
        "status": "active",
    }

    if table_type != "Monitoring_Output" or datalake_env == "delta":
        data["dbfs_path"] = table_path
        data["datalake_env"] = "delta"
    else:
        data["db_path"] = f"{gcp_project_id}.{bq_database_name}.{table_path}"
        data["datalake_env"] = datalake_env

    # Calling tables add API
    tables_add(data)


def throw_exception(e):
    """
    Updates job run and job task as failed upon occurence of an exception
    """
    stagemetrics.end()
    taskmetrics.end()

    stage_Df = stagemetrics.create_stagemetrics_DF("PerfStageMetrics")
    task_Df = taskmetrics.create_taskmetrics_DF("PerfTaskMetrics")

    aggregate_compute_metrics = (
        stagemetrics.aggregate_stagemetrics_DF()
        .select("executorCpuTime", "peakExecutionMemory")
        .collect()[0]
        .asDict()
    )

    aggregate_compute_metrics["executorCpuTime"] = (
        aggregate_compute_metrics["executorCpuTime"] / 1000
        if aggregate_compute_metrics["executorCpuTime"]
        else 0
    )
    aggregate_compute_metrics["peakExecutionMemory"] = (
        aggregate_compute_metrics["peakExecutionMemory"] / (1024 * 1024)
        if aggregate_compute_metrics["peakExecutionMemory"]
        else 0
    )

    compute_metrics = {
        "stagemetrics": stage_Df.rdd.map(lambda row: row.asDict()).collect(),
        "taskmetrics": task_Df.rdd.map(lambda row: row.asDict()).collect(),
    }

    utils.log("exception", message_run)
    utils.log(str(e), message_run)
    ts = str(int(time.time() * 1000000))
    run_id, job_id, table, source = fetch_id()
    run_notebook_url = generate_run_notebook_url(job_id, run_id)
    model_inference_info = dbutils.widgets.get("model_inference_info")
    if isinstance(model_inference_info, str):
        model_inference_info = json.loads(model_inference_info)
    project_id, version = (
        model_inference_info["project_id"],
        model_inference_info["version"],
    )
    created_by_id, created_by_name = (
        model_inference_info["created_by_id"],
        model_inference_info["created_by_name"],
    )

    # Updating job run
    message_run.append(
        {
            "time": ts,
            "message": "Monitor Job with job id: "
            + str(job_id)
            + " and run id: "
            + str(run_id)
            + " failed with exception- "
            + str(e)
            + ".",
        }
    )
    message_task.append(
        {
            "time": ts,
            "message": "task0"
            + " within job id: "
            + str(job_id)
            + " and run id: "
            + str(run_id)
            + " failed with exception - "
            + str(e)
            + ".",
        }
    )
    log_data = {
        "project_id": project_id,
        "version": str(version),
        "job_id": str(job_id),
        "run_id": str(run_id),
        "end_time": str(ts),
        "status": "failed",
        "message": message_run,
        "updated_by_id": created_by_id,
        "updated_by_name": created_by_name,
        "task_id": "task0",
        "job_type": "Monitor",
        "cpu": str(aggregate_compute_metrics.get("executorCpuTime", "NA")),
        "ram": str(aggregate_compute_metrics.get("peakExecutionMemory", "NA")),
        "cluster_info": get_cluster_info(spark),
        "compute_metrics": compute_metrics,
        "run_notebook_url": run_notebook_url,
        "deployment_master_id": deployment_master_id,
    }
    response = requests.put(API_ENDPOINT + JOB_RUNS_UPDATE, json=log_data, headers=h1)
    print(
        f"\n\
    Logging task:\n\
    endpoint - {JOB_RUNS_UPDATE}\n\
    status   - {response}\n\
    response - {response.text}\n\
    payload  - {log_data}\n"
    )

    utils.save_sparkmeasure_aggregated_tables(
        project_name=project_name,
        job_run_update_payload=log_data,
        compute_usage_metrics=aggregate_compute_metrics,
        stagemetrics=stagemetrics,
        taskmetrics=taskmetrics,
        api_endpoint=API_ENDPOINT,
        headers=h1,
        spark=spark,
        dbutils=dbutils,
    )

    # Updating job task
    task_log_data = {
        "job_id": str(job_id),
        "run_id": str(run_id),
        "task_id": "task0",
        "end_time": str(ts),
        "status": "failed",
        "message": message_task,
        "updated_by_id": created_by_id,
        "updated_by_name": created_by_name,
        "job_type": "Monitor",
        "run_notebook_url": run_notebook_url,
    }
    response = requests.put(
        API_ENDPOINT + JOB_TASK_UPDATE, json=task_log_data, headers=h1
    )

    print(
        f"\n\
    Logging task:\n\
    endpoint - {JOB_TASK_UPDATE}\n\
    status   - {response}\n\
    response - {response.text}\n\
    payload  - {task_log_data}\n"
    )

    cancel_job_run_data = {"run_id": str(run_id), "status": "failed"}
    response = requests.post(
        API_ENDPOINT + CANCEL_JOB_RUN, json=cancel_job_run_data, headers=h1
    )

    print(
        f"\n\
    Logging task:\n\
    endpoint - {CANCEL_JOB_RUN}\n\
    status   - {response}\n\
    response - {response.text}\n\
    payload  - {cancel_job_run_data}\n"
    )

    # Exiting the job
    dbutils.notebook.exit(e)


def register_delta_as_hive(db_name, table_name, dbfs_path, spark):
    spark.sql(f"CREATE DATABASE IF NOT EXISTS {db_name};")
    spark.sql(f"USE {db_name};")
    spark.sql(
        f"""
        CREATE EXTERNAL TABLE IF NOT EXISTS {table_name}
        USING DELTA
        LOCATION '{dbfs_path}';
    """
    )


@retry(
    wait=wait_exponential(min=4, multiplier=1, max=10),
    stop=(stop_after_delay(10) | stop_after_attempt(5)),
)
def get_model_details(model_artifact_id, h1):
    model_details_response = requests.get(
        f"{API_ENDPOINT}{GET_MODEL_ARTIFACT_API}/get?model_artifact_id={model_artifact_id}",
        headers=h1,
    )
    utils.log(
        f"\n\
    endpoint - {GET_MODEL_ARTIFACT_API}/get?model_artifact_id={model_artifact_id}\n\
    response - {model_details_response}",
        message_run,
    )
    if model_details_response.status_code not in [200, 201]:
        raise Exception(
            f"The {GET_MODEL_ARTIFACT_API} returned :{model_details_response.status_code} response."
        )

    model_details_json = model_details_response.json()
    utils.log(f"MODEL DETAILS: {model_details_json}", message_run)
    return model_details_json["data"][0]


def read_data(data_path, is_platform_table=False):
    env_to_read = datalake_env if not is_platform_table else platform_datalake_env
    if env_to_read == "delta":
        return utils.df_read(
            data_path=data_path, spark=spark, resource_type=env_to_read
        )
    else:
        return utils.df_read(
            spark=spark,
            data_path=data_path.split(".")[-1],
            bq_database_name=bq_database_name,
            bq_project_id=gcp_project_id,
            encrypted_service_account=encrypted_sa_details,
            encryption_key=encryption_key,
            resource_type=env_to_read,
        )


def write_data(data_path, dataframe, mode, partition_by, is_platform_table=False):
    env_to_write = datalake_env if not is_platform_table else platform_datalake_env
    if env_to_write == "delta":
        utils.df_write(
            data_path=data_path,
            dataframe=dataframe,
            mode=mode,
            resource_type=env_to_write,
            partition_by=partition_by,
        )
    else:
        utils.df_write(
            data_path=data_path,
            dataframe=dataframe,
            mode=mode,
            bucket_name=f"{az_container_name}_{env}",
            bq_database_name=bq_database_name,
            bq_project_id=gcp_project_id,
            encrypted_service_account=encrypted_sa_details,
            encryption_key=encryption_key,
            resource_type=env_to_write,
            partition_by=partition_by,
        )


def table_already_created(data_path, is_agg_table=False):
    if is_agg_table:
        env_to_write = platform_datalake_env
        is_platform_table = True
    else:
        env_to_write = datalake_env
        is_platform_table = False
    utils.log(f"{env_to_write}", message_run)
    if env_to_write == "delta":
        return DeltaTable.isDeltaTable(spark, data_path)
    elif env_to_write == "bigquery":
        try:
            read_data(data_path, is_platform_table).first()
            return True
        except Exception as e:
            utils.log(str(e), message_run)
            return False


def find_integer(obj):
    if isinstance(obj, dict):  # If the input is a dictionary
        for value in obj.values():
            result = find_integer(value)  # Recursively search for an integer
            if result is not None:  # If an integer is found, return it
                return result
    elif isinstance(obj, list):  # If the input is a list
        for item in obj:
            result = find_integer(item)  # Recursively search each item for an integer
            if result is not None:  # If an integer is found, return it
                return result
    elif isinstance(obj, pd.core.series.Series):
        obj = obj.to_list()
        result = find_integer(obj)
        if result is not None:  # If an integer is found, return it
            return result
    elif isinstance(obj, int):  # If the input is an integer
        return obj  # Return the integer
    return None  # If no integer is found, return None


# Handling a JSON string input
def extract_integer(input):
    if isinstance(input, str):  # Check if the input is a string
        try:
            input = json.loads(input)  # Try to parse the JSON string
        except json.JSONDecodeError:
            return None  # Return None if the string cannot be parsed
    return find_integer(input)  # Use the find_integer function to extract the integer


# COMMAND ----------

# MAGIC %md
# MAGIC ## Monitoring Sub-Type based execution Methods

# COMMAND ----------

dbutils.widgets.text("job_id","")

# COMMAND ----------

dbutils.widgets.text("run_id","")

# COMMAND ----------


def update_task_log_as_per_inference_table(monitoring_subtype, required_entry):
    """
    Updates the task log table for monitoring subtype
    """
    # Loading and filtering task log for the given monitoring subtype
    # task_log = (
    #             spark.read.load(task_log_path)
    #             .filter(F.col('monitoring_subtype') == monitoring_subtype)
    #             .orderBy('date', 'timestamp', ascending=False)
    # )

    # # Loading inference task log
    # inference_task_log_path = f"dbfs:/mnt/{az_container_name}/{env}/{project_id}/{version}/{model_inference_info['parent_model_infer_job_id']}/task_log_table"
    # df_inf_task = spark.read.load(inference_task_log_path)

    # # Determination of start marker and end marker
    # df_inf_task = (
    #                 df_inf_task
    #                 .filter(F.col("start_marker") > required_entry['end_marker'])
    #                 .orderBy('date', 'timestamp', ascending=True)
    #             )
    # if df_inf_task.first() != None:
    #     start_marker = df_inf_task.first()['start_marker']
    #     end_marker = df_inf_task.first()['end_marker']
    # else:
    if isinstance(required_entry, pyspark.sql.dataframe.DataFrame):
        required_entry = required_entry.toPandas()
    start_marker = extract_integer(required_entry["start_marker"])
    end_marker = extract_integer(required_entry["end_marker"])
    utils.log(f"start_marker : {start_marker}", message_run)
    utils.log(f"end_marker : {end_marker}", message_run)
    # Determination of table name on which markers have been calculated
    table_name = "inference_table"

    # Updating task log with new metadata
    ts = int(time.time() * 1000000)
    schema = StructType(
        [
            StructField("monitoring_subtype", StringType(), True),
            StructField("start_marker", IntegerType(), True),
            StructField("end_marker", IntegerType(), True),
            StructField("table_name", StringType(), True),
        ]
    )
    df_column_name = ["monitoring_subtype", "start_marker", "end_marker", "table_name"]
    df_record = [(monitoring_subtype, int(start_marker), int(end_marker), table_name)]
    df_task = spark.createDataFrame(df_record, schema=schema)
    df_task = df_task.withColumn(
        "timestamp",
        F.expr("reflect('java.lang.System', 'currentTimeMillis')").cast("long"),
    )
    df_task = df_task.withColumn("date", F.lit(date))
    df_task = df_task.withColumn("date", F.to_date(F.col("date"), "MM-dd-yyyy"))
    df_task = df_task.withColumn("job_id", F.lit(job_id))
    df_task = df_task.withColumn("run_id", F.lit(run_id))
    if model_inference_info.get("inference_task_log_table_id", None):
        try:
            df_task = df_task.withColumn("inference_id", F.lit(required_entry["id"]))
        except:
            pass

    # Adding id column
    try:
        existing_df_task = read_data(task_log_path, True)
        max_id = int(existing_df_task.select(F.max("id")).first()[0])
    except:
        max_id = 1

    try:
        df_task = df_task.withColumn("id", F.lit(max_id + 1))
        write_data(
            data_path=task_log_path,
            dataframe=df_task,
            mode="append",
            partition_by=None,
            is_platform_table=True,
        )
    except:
        if "id" in df_task.columns:
            df_task = df_task.drop("id")
        write_data(
            data_path=task_log_path,
            dataframe=df_task,
            mode="append",
            partition_by=None,
            is_platform_table=True,
        )

    print(df_task.display())


# COMMAND ----------

# MAGIC %md
# MAGIC ## Monitoring Specific Helper Methods

# COMMAND ----------


# Function to get numeric columns
def list_numerical_columns(data):
    schema = data.dtypes
    numerical_cols = [
        x[0] for x in schema if x[1] not in ["string", "date", "boolean", "timestamp"]
    ]
    return numerical_cols


def list_datelike_columns(data):
    schema = data.dtypes
    date_cols = [x[0] for x in schema if x[1] in ["date", "timestamp"]]
    return date_cols


# Algo ID to name mapper
def map_algo_id_2_name(algo_id_list):
    algo_name_list = []
    for id in algo_id_list:
        if id == "DD1":
            name = "lsdd"
        elif id == "DD2":
            name = "mmd"
        elif id == "DD3":
            name = "cammd"
        elif id == "DD4":
            name = "lkmmd"
        elif id == "DD5":
            name = "spotdiff"
        elif id == "DD6":
            name = "ks"
        elif id == "DD7":
            name = "emperical_mmd"
        elif id == "DD8":
            name = "chisquare"
        elif id == "DD9":
            name = "fet_ev"
        elif id == "DD10":
            name = "cvm_ev"
        elif id == "FD1":
            name = "mtd"
        elif id == "FD2":
            name = "psi"
        elif id == "FD3":
            name = "acf"
        elif id == "FD4":
            name = "additive-ts-decompose"
        elif id == "FD5":
            name = "pacf"
        elif id == "FD6":
            name = "js"
        elif id == "FD7":
            name = "anderson"
        elif id == "FD8":
            name = "psi_ev"
        elif id == "FD9":
            name = "hellinger"
        elif id == "FD10":
            name = "mann_witney"
        elif id == "FD11":
            name = "ks"
        elif id == "FD12":
            name = "kl_div"
        elif id == "FD13":
            name = "chisquare"
        elif id == "FD14":
            name = "multiplicative-ts-decompose"
        elif id == "FD15":
            name = "z_test"
        elif id == "FD16":
            name = "wasserstein"
        elif id == "FD17":
            name = "fet_ev"
        elif id == "FD18":
            name = "cvm_ev"
        elif id == "FD19":
            name = "g_test"
        elif id == "FD20":
            name = "energy_distance"
        elif id == "FD21":
            name = "epps_singleton"
        elif id == "FD22":
            name = "t_test"
        elif id == "FD23":
            name = "emperical_mmd"
        elif id == "FD24":
            name = "tvd"
        elif id == "TD1":
            name = "lsdd"
        elif id == "TD2":
            name = "mmd"
        elif id == "TD3":
            name = "mtd"
        elif id == "TD4":
            name = "js"
        elif id == "TD5":
            name = "anderson"
        elif id == "TD6":
            name = "psi_ev"
        elif id == "TD7":
            name = "hellinger"
        elif id == "TD8":
            name = "mann_witney"
        elif id == "TD9":
            name = "ks"
        elif id == "TD10":
            name = "kl_div"
        elif id == "TD11":
            name = "chisquare"
        elif id == "TD12":
            name = "additive-ts-decompose"
        elif id == "TD13":
            name = "multiplicative-ts-decompose"
        elif id == "TD14":
            name = "acf"
        elif id == "TD15":
            name = "pacf"
        elif id == "TD16":
            name = "z_test"
        elif id == "TD17":
            name = "wasserstein"
        elif id == "TD18":
            name = "fet_ev"
        elif id == "TD19":
            name = "cvm_ev"
        elif id == "TD20":
            name = "g_test"
        elif id == "TD21":
            name = "energy_distance"
        elif id == "TD22":
            name = "epps_singleton"
        elif id == "TD23":
            name = "t_test"
        elif id == "TD24":
            name = "emperical_mmd"
        elif id == "TD25":
            name = "tvd"
        elif id == "CD1":
            name = "fet"
        elif id == "CD2":
            name = "cvm"
        elif id == "CD3":
            name = "fet_ev"
        elif id == "CD4":
            name = "cvm_ev"
        elif id == "r2":
            name = "r2"
        elif id == "mse":
            name = "mse"
        elif id == "mae":
            name = "mae"
        elif id == "rmse":
            name = "rmse"
        elif id == "accuracy":
            name = "accuracy"
        elif id == "f1":
            name = "f1"
        elif id == "log_loss":
            name = "log_loss"
        elif id == "precision":
            name = "precision"
        elif id == "recall":
            name = "recall"
        elif id == "roc_auc":
            name = "roc_auc"
        elif id == "smape":
            name = "smape"
        elif id == "mdape":
            name = "mdape"
        elif id == "directional_accuracy":
            name = "mda"
        elif id == "mape":
            name = "mape"
        else:
            name = None
        algo_name_list.append(name)
    return algo_name_list


# ModelType ID to name mapper
def map_modeltype_id_2_name(model_id):
    if model_id == "Regression":
        return "regression"
    if model_id == "Classification":
        return "classification"
    if model_id == "Forecasting":
        return "forecasting"
    if model_id == "MMM":
        return "mmm"


# Function to round monitoring metrics
def round_metrics(data):
    if isinstance(data, dict):
        for key in data.keys():
            data[key] = round_metrics(data[key])
    elif isinstance(data, (list, tuple)):
        for i, val in enumerate(data):
            data[i] = round_metrics(val)
    elif isinstance(data, (int, float)):
        data = round(data, 5)
    return data


# Function to handle nan and None in JSON so that the JSON is complaint
def json_encode_nans(data: dict):
    """Custom Encoders are needed when json.dumps doesnt know how to handle some dtypes.
    But on this case np.nan and Nones are something json.dumps knows how to handle.
    So we cannot handle these tupes in custom JSON encoders.
    To override nans and Nones, this recursive function is written"""

    if isinstance(data, dict):
        for key in data.keys():
            data[key] = json_encode_nans(data[key])
    elif isinstance(data, (list, tuple)):
        for i, val in enumerate(data):
            data[i] = json_encode_nans(val)
    elif (str(data) in ["nan", "inf", "-inf"]) or (data is None):
        data = ""
    return data


# Function to check for monitor setup
def check_for_monitor_setup(drift_conf, target_type=None):
    monitor_algo_to_setup = []
    listed_monitors = []
    master_algo_list = []

    if drift_conf["monitoring_type"] == "feature_drift":
        feature_algo_list = drift_conf["monitoring_algo_features"]
        master_algo_list.append(feature_algo_list)
        overall_algo_list = drift_conf["monitoring_algo_overall"]
        master_algo_list.append(overall_algo_list)
        monitoring_type = drift_conf["monitoring_type"]
    elif drift_conf["monitoring_type"] == "target_drift":
        target_algo_list = drift_conf["monitoring_algo"]
        master_algo_list.append(target_algo_list)
        monitoring_type = drift_conf["monitoring_type"] + "_" + target_type
    elif drift_conf["monitoring_type"] == "concept_drift":
        concept_algo_list = drift_conf["monitoring_algo"]
        master_algo_list.append(concept_algo_list)
        monitoring_type = drift_conf["monitoring_type"]
    elif drift_conf["monitoring_type"] == "performance_drift":
        performance_algo_list = drift_conf["monitoring_metrics"]
        master_algo_list.append(performance_algo_list)
        monitoring_type = drift_conf["monitoring_type"]
    try:
        for algo_list in master_algo_list:
            for algo in algo_list:
                algo_name = map_algo_id_2_name([algo["algo_id"]])[0]
                monitoring_artifact_path_for_algo = generate_monitoring_artifact_path(
                    monitoring_type=monitoring_type,
                    monitoring_artifacts_base_dir=monitoring_artifact_path,
                    monitor_algo=algo_name,
                )
                listed_monitors.append(algo_name)
                utils.log(f"{monitoring_artifact_path_for_algo}", message_run)
                if os.path.exists(monitoring_artifact_path_for_algo) and os.path.isdir(
                    monitoring_artifact_path_for_algo
                ):
                    # Artifacts are already generated, proceed to monitor
                    pass
                else:
                    # Add to list of algorithms to setup
                    monitor_algo_to_setup.append(algo_name)

        utils.log(f"listed_feature_monitors : {listed_monitors}", message_run)
        utils.log(
            f"feature_monitor_algo_to_setup : {monitor_algo_to_setup}", message_run
        )
    except Exception as e:
        traceback.print_exc()

    return monitor_algo_to_setup, listed_monitors


# Fuction to setup monitor
def setup_monitor(
    monitor_algo_to_setup,
    drift_conf,
    data_type,
    monitoring_artifact_path,
    reference_df=None,
    target=None,
    target_type=None,
    base_y_actuals=None,
    base_y_predicted=None,
    date_column=None,
):
    artifact_path = "No Artifacts Created"

    # Sample size selection
    DATA_DRIFT_SAMPLE = 1000
    MODEL_DRIFT_SAMPLE = 1000

    if not monitor_algo_to_setup:
        utils.log(f"artifact_path : {artifact_path}", message_run)
        return
    try:
        if drift_conf["monitoring_type"] == "feature_drift":
            if reference_df.shape[0] < 1000:
                DATA_DRIFT_SAMPLE = reference_df.shape[0]
            artifact_path = setup_feature_drift_monitors(
                data_type=data_type,
                reference_df=reference_df.sample(n=DATA_DRIFT_SAMPLE),
                monitoring_artifacts_base_dir=monitoring_artifact_path,
                monitored_features=drift_conf["monitored_features"],
                monitoring_algos=monitor_algo_to_setup,
            )
        elif drift_conf["monitoring_type"] == "target_drift":
            if drift_conf["monitoring_mode"] == "Batch":
                if reference_df.shape[0] < 1000:
                    DATA_DRIFT_SAMPLE = reference_df.shape[0]
                artifact_path = setup_target_drift_monitors(
                    data_type=data_type,
                    reference_df=reference_df.sample(n=DATA_DRIFT_SAMPLE),
                    target=target,
                    target_type=target_type,
                    monitoring_artifacts_base_dir=monitoring_artifact_path,
                    monitoring_algos=monitor_algo_to_setup,
                    date_column=date_column,
                )
        elif drift_conf["monitoring_type"] == "concept_drift":
            if drift_conf["monitoring_mode"] == "Batch":
                if base_y_actuals.shape[0] < 1000:
                    MODEL_DRIFT_SAMPLE = base_y_actuals.shape[0]
                artifact_path = setup_concept_drift_monitors(
                    data_type=data_type,
                    model_task=map_modeltype_id_2_name(
                        drift_conf["modelling_task_type"]
                    ),
                    base_y_actuals=base_y_actuals[:MODEL_DRIFT_SAMPLE],
                    base_y_predicted=base_y_predicted[:MODEL_DRIFT_SAMPLE],
                    monitoring_algos=monitor_algo_to_setup,
                    monitoring_artifacts_base_dir=monitoring_artifact_path,
                )
        elif drift_conf["monitoring_type"] == "performance_drift":
            if drift_conf["monitoring_mode"] == "Batch":
                artifact_path = setup_performance_drift_monitors(
                    data_type=data_type,
                    model_task=map_modeltype_id_2_name(
                        drift_conf["modelling_task_type"]
                    ),
                    base_y_actuals=base_y_actuals,
                    base_y_predicted=base_y_predicted,
                    monitoring_metrics=monitor_algo_to_setup,
                    monitoring_artifacts_base_dir=monitoring_artifact_path,
                )
        utils.log(f"artifact_path : {artifact_path}", message_run)
    except Exception as e:
        traceback.print_exc()


# Function to monitor current data
def monitor_current_data(
    drift_conf,
    data_type,
    monitoring_artifact_path,
    monitoring_algos,
    current_df=None,
    target=None,
    target_type=None,
    current_y_actuals=None,
    current_y_predicted=None,
    date_column=None,
):
    drift_metrics_list = []
    try:
        if (current_df is not None) and (current_df.shape[0] == 0):
            raise Exception("There is no current data to monitor")
        elif (current_y_actuals is not None) and (current_y_actuals.shape[0] == 0):
            raise Exception("There is no current data to monitor")
        elif (current_y_predicted is not None) and (current_y_predicted.shape[0] == 0):
            raise Exception("There is no current data to monitor")

        # Sample size selection
        DATA_DRIFT_SAMPLE = 1000
        MODEL_DRIFT_SAMPLE = 1000

        if drift_conf["monitoring_type"] == "feature_drift":
            if drift_conf["monitoring_mode"] == "Batch":
                if current_df.shape[0] < 1000:
                    DATA_DRIFT_SAMPLE = current_df.shape[0]
                drift_metrics_list = monitor_feature_drift(
                    data_type=data_type,
                    current_df=current_df.sample(n=DATA_DRIFT_SAMPLE),
                    monitoring_artifacts_base_dir=monitoring_artifact_path,
                    monitored_features=drift_conf["monitored_features"],
                    monitoring_algos=monitoring_algos,
                    monitoring_algo_thresholds=drift_conf[
                        "monitoring_algo_feature_thresholds"
                    ]
                    + drift_conf["monitoring_algo_overall_thresholds"],
                )
        elif drift_conf["monitoring_type"] == "target_drift":
            if drift_conf["monitoring_mode"] == "Batch":
                if current_df.shape[0] < 1000:
                    DATA_DRIFT_SAMPLE = current_df.shape[0]
                drift_metrics_list = monitor_target_drift(
                    data_type=data_type,
                    current_df=current_df.sample(n=DATA_DRIFT_SAMPLE),
                    target=target,
                    target_type=target_type,
                    monitoring_artifacts_base_dir=monitoring_artifact_path,
                    monitoring_algos=monitoring_algos,
                    monitoring_algo_thresholds=drift_conf["monitoring_algo_thresholds"],
                    date_column=date_column,
                )
        elif drift_conf["monitoring_type"] == "concept_drift":
            if drift_conf["monitoring_mode"] == "Batch":
                if current_y_actuals.shape[0] < 1000:
                    MODEL_DRIFT_SAMPLE = current_y_actuals.shape[0]
                drift_metrics_list = monitor_concept_drift(
                    data_type=data_type,
                    current_y_actuals=current_y_actuals[:MODEL_DRIFT_SAMPLE],
                    current_y_predicted=current_y_predicted[:MODEL_DRIFT_SAMPLE],
                    monitoring_algos=monitoring_algos,
                    monitoring_artifacts_base_dir=monitoring_artifact_path,
                    monitoring_algo_thresholds=drift_conf["monitoring_algo_thresholds"],
                )
                for metric in drift_metrics_list:
                    metric["stat_val"] = metric["stat_val"][0]
                    metric["p_val"] = metric["p_val"][0]
        elif drift_conf["monitoring_type"] == "performance_drift":
            if drift_conf["monitoring_mode"] == "Batch":
                drift_json = monitor_performance_drift(
                    data_type=data_type,
                    model_task=map_modeltype_id_2_name(
                        drift_conf["modelling_task_type"]
                    ),
                    current_y_actuals=current_y_actuals,
                    current_y_predicted=current_y_predicted,
                    monitoring_metrics=monitoring_algos,
                    monitoring_artifacts_base_dir=monitoring_artifact_path,
                    monitoring_metrics_thresholds=drift_conf[
                        "monitoring_algo_thresholds"
                    ][0],
                )
                drift_metrics_list = [drift_json]
    except Exception as e:
        traceback.print_exc()

    return drift_metrics_list


def delta_encode_str(data: dict):
    if isinstance(data, dict):
        for key in data.keys():
            data[key] = delta_encode_str(data[key])
    elif isinstance(data, (list, tuple)):
        for i, val in enumerate(data):
            data[i] = delta_encode_str(val)
    elif str(data) in [" ", "", "", " "]:
        data = None
    return data


# COMMAND ----------

# MAGIC %md
# MAGIC ## Metrics write into delta Helper Methods

# COMMAND ----------


def get_metric_table_schema(kind):
    """
    Returns the predefined schema for the dedicated table for the kind of monitoring metrics to be written
    """
    if kind != "performance_drift":
        data_model_schema = StructType(
            [
                StructField("feature_attribute", StringType(), True),
                StructField("is_drift", LongType(), True),
                StructField("stat_val", DoubleType(), True),
                StructField("p_val", DoubleType(), True),
                StructField("model_artifact_id", StringType(), True),
                StructField("input_table_id", StringType(), True),
                StructField("data_prep_deployment_job_id", StringType(), True),
                StructField("deployment_master_id", StringType(), True),
                StructField("processing_time", LongType(), True),
                StructField("batch_start_id", StringType(), True),
                StructField("batch_end_id", StringType(), True),
                StructField("batch_size", LongType(), True),
                StructField("monitoring_type", StringType(), True),
                StructField("monitoring_sub_type", StringType(), True),
                StructField("env", StringType(), True),
                StructField("created_by_name", StringType(), True),
                StructField("created_by_id", StringType(), True),
                StructField(
                    "batch_markers", MapType(StringType(), StringType(), True), True
                ),
                StructField("monitoring_algo_id", StringType(), True),
                StructField("monitoring_algo_name", StringType(), True),
                StructField("job_id", StringType(), True),
                StructField("run_id", StringType(), True),
                StructField("modelling_task_name", StringType(), True),
                StructField("model_name", StringType(), True),
                StructField("model_version", StringType(), True),
                StructField("model_session_id", StringType(), True),
                StructField("project_id", StringType(), True),
                StructField("project_name", StringType(), True),
                StructField("version", StringType(), True),
                StructField("inference_metadata", StringType(), True),
                StructField("granularity", StringType(), True),
            ]
        )
        return data_model_schema

    else:
        performance_data_schema = StructType(
            [
                StructField("feature_attribute", StringType(), True),
                StructField("is_drift", LongType(), True),
                StructField("value", DoubleType(), True),
                StructField("model_artifact_id", StringType(), True),
                StructField("input_table_id", StringType(), True),
                StructField("data_prep_deployment_job_id", StringType(), True),
                StructField("deployment_master_id", StringType(), True),
                StructField("processing_time", LongType(), True),
                StructField("batch_start_id", StringType(), True),
                StructField("batch_end_id", StringType(), True),
                StructField("batch_size", LongType(), True),
                StructField("monitoring_type", StringType(), True),
                StructField("monitoring_sub_type", StringType(), True),
                StructField("env", StringType(), True),
                StructField("created_by_name", StringType(), True),
                StructField("created_by_id", StringType(), True),
                StructField(
                    "batch_markers", MapType(StringType(), StringType(), True), True
                ),
                StructField("modelling_task_name", StringType(), True),
                StructField("monitoring_algo_name", StringType(), True),
                StructField("job_id", StringType(), True),
                StructField("run_id", StringType(), True),
                StructField("model_name", StringType(), True),
                StructField("model_version", StringType(), True),
                StructField("model_session_id", StringType(), True),
                StructField("project_id", StringType(), True),
                StructField("project_name", StringType(), True),
                StructField("version", StringType(), True),
                StructField("inference_metadata", StringType(), True),
                StructField("granularity", StringType(), True),

            ]
        )
        return performance_data_schema


# COMMAND ----------

# MAGIC %md
# MAGIC ## Global Variables

# COMMAND ----------

dbutils.widgets.text("model_inference_info", "")

# COMMAND ----------

model_inference_info = dbutils.widgets.get("model_inference_info")

# COMMAND ----------

print(model_inference_info)

# COMMAND ----------

# Fetching env and scope information
env, vault_scope = get_env_vault_scope()
az_container_name = str(dbutils.secrets.get(scope=vault_scope, key=SECRET_MAPPING.get("az-container-name","")))

# API Endpoints
# API_ENDPOINT = "https://mlcoredevv2pg21.azurewebsites.net/"
API_ENDPOINT = "https://mlcoredevv2sql21test.azurewebsites.net/"
SCHEDULE_STATUS = "mlapi/task/schedule/status"
JOB_TASK_ADD = "mlapi/job/task/log/add"
TABLES_ADD = "mlapi/tables/add"
JOB_TASK_UPDATE = "mlapi/job/task/log/update"
JOB_RUNS_UPDATE = "mlapi/job/runs/log/update"
TRANSFORMS_SCHEMA = "mlapi/transforms/schema/list"
EDA_RUN = "mlapi/eda/run"
LIST_TABLES = "mlapi/tables/list"
TABLES_UPDATE = "mlapi/tables/update"
CANCEL_JOB_RUN = "mlapi/jobs/run/cancel"
TASKS_STATUS = "mlapi/job/run/tasks/status"
RUN_TASK_ADD = "mlapi/job/runs/log/add"
JOB_LOGS = "mlapi/jobs_logs/list"
GET_MODEL_ARTIFACT_API = "mlapi/modelartifact"
TABLES_METADATA = "mlapi/tables/metadata/list"
message_run = []
message_task = []

# Text String to get the API
h1 = get_headers(vault_scope)

# Notebook Params
# run_id, job_id, table, source = fetch_id()
job_id = dbutils.widgets.get("job_id")
run_id = dbutils.widgets.get("run_id")
run_notebook_url = generate_run_notebook_url(job_id, run_id)
model_inference_info = dbutils.widgets.get("model_inference_info")
if isinstance(model_inference_info, str):
    model_inference_info = json.loads(model_inference_info)

project_id, version = (
    model_inference_info["project_id"],
    model_inference_info["version"],
)
created_by_id, created_by_name = (
    model_inference_info["created_by_id"],
    model_inference_info["created_by_name"],
)
model_artifact_id = model_inference_info["model_artifact_id"]
model_name = model_inference_info["model_name"]
# Reference and Current table metadata
reference_table_path = model_inference_info["reference_table"]["path"]
current_table_path = model_inference_info["current_table"]["path"]
reference_table_id = model_inference_info["reference_table"]["table_id"]
current_table_id = model_inference_info["current_table"]["table_id"]
inference_table_path = model_inference_info["inference_output_table_path"]
yactuals_table_path = model_inference_info["y_actual_table_path"]
deployment_master_id = model_inference_info["deployment_master_id"]
db_name = f"{project_id}_{version}"
project_name = model_inference_info.get("project_name", "")

if not inference_table_path.startswith("dbfs:/") and inference_table_path:
    inference_table_path = f"dbfs:/user/hive/warehouse/{inference_table_path.split('.')[-2]}.db/{inference_table_path.split('.')[-1]}"

# deployment env
now = datetime.now()
date = now.strftime("%m-%d-%Y")

try:
    batch_size = int(get_params("batch_size"))
    if batch_size == 0:
        batch_size = 10000
except:
    batch_size = 10000

try:
    env = dbutils.widgets.get("env")
except:
    env = "dev"

try:
    deployment_env = dbutils.widgets.get("deployment_env")
except:
    deployment_env = "dev"

try:
    datalake_env = dbutils.widgets.get("datalake_env")
except Exception as e:
    utils.log(f"Exception while retrieving data lake environment : {e}", message_run)
    datalake_env = "delta"

primary_keys = model_inference_info.get("primary_keys", [])
if primary_keys in ["", None]:
    primary_keys = []
if isinstance(primary_keys, str):
    primary_keys = json.loads(primary_keys)

target_columns = model_inference_info.get("target_columns", [])
if target_columns in ["", None]:
    target_columns = []
if isinstance(target_columns, str):
    try:
        target_columns = json.loads(target_columns)
    except:
        try:
            target_columns = json.loads(target_columns.replace("'", '"'))
        except:
            target_columns = ast.literal_eval(target_columns)

platform_datalake_env = dbutils.secrets.get(vault_scope, SECRET_MAPPING.get("platform-datalake-env",""))

# Defining inference task log path
if model_inference_info.get("inference_task_log_table_id", None):
    inference_task_log_path = get_table_dbfs_path(
        "", "", model_inference_info["inference_task_log_table_id"]
    )
else:
    if platform_datalake_env.lower() == "delta":
        inference_task_log_path = f"dbfs:/mnt/{az_container_name}/{env}/{project_id}/{version}/{model_inference_info['parent_model_infer_job_id']}/task_log_table"
    else:
        encrypted_sa_details = dbutils.secrets.get(vault_scope, SECRET_MAPPING.get("gcp-service-account-encypted",""))
        encryption_key = dbutils.secrets.get(vault_scope, SECRET_MAPPING.get("gcp-service-account-private-key",""))
        bq_database_name = dbutils.secrets.get(vault_scope, SECRET_MAPPING.get("gcp-bq-database-name",""))
        gcp_project_id = dbutils.secrets.get(vault_scope, SECRET_MAPPING.get("gcp-api-quota-project-id",""))

        inference_task_log_path = f"{env}_{project_id}_{version}_{model_inference_info['parent_model_infer_job_id']}_task_log_table"


# COMMAND ----------

# # Checking if upstream jobs, i.e. DPD and Model Inference, are active or not, skipping the run if not active.
# data_prep_deployment_job_id = model_inference_info.get(
#     "data_prep_deployment_job_id", ""
# )
# model_inference_job_id = model_inference_info.get("parent_model_infer_job_id", "")

# job_ids_to_check = []
# if data_prep_deployment_job_id != "" and data_prep_deployment_job_id != None:
#     job_ids_to_check.append(data_prep_deployment_job_id)

# if model_inference_job_id != "" and model_inference_job_id != None:
#     job_ids_to_check.append(model_inference_job_id)

# if not check_if_upstream_job_active(job_ids_to_check):
#     declare_job_as_successful(upstream_inactive=True)

# COMMAND ----------

try:
    # Updating the task log info
    ts = str(int(time.time() * 1000000))
    log_data = {
        "project_id": project_id,
        "version": version,
        "job_id": str(job_id),
        "run_id": str(run_id),
        "status": "running",
        "start_time": str(ts),
        "created_by_id": created_by_id,
        "created_by_name": created_by_name,
        "job_type": "Monitor",
        "project_name": project_name,
        "deployment_master_id": deployment_master_id,
        "run_notebook_url": run_notebook_url,
    }
    response = requests.post(API_ENDPOINT + RUN_TASK_ADD, json=log_data, headers=h1)
    utils.log(
        f"\n\
    Logging task:\n\
    endpoint - {RUN_TASK_ADD}\n\
    status   - {response}\n\
    payload  - {log_data}",
        message_run,
    )

    task_log_data = {
        "project_id": project_id,
        "version": version,
        "job_id": str(job_id),
        "run_id": str(run_id),
        "task_id": "task0",
        "status": "running",
        "start_time": str(ts),
        "created_by_id": created_by_id,
        "created_by_name": created_by_name,
        "job_type": "Monitor",
    }
    response = requests.post(
        API_ENDPOINT + JOB_TASK_ADD, json=task_log_data, headers=h1
    )
    utils.log(
        f"\n\
    Logging task:\n\
    endpoint - {JOB_TASK_ADD}\n\
    status   - {response}\n\
    response - {response.text}\n\
    payload  - {task_log_data}\n",
        message_run,
    )

    # Updating message run and message task
    t = str(int(time.time() * 1000000))
    message_run.append({"time": t, "message": "Monitor Task has been scheduled."})
    message_task.append({"time": t, "message": "Monitor Task has been scheduled."})

except Exception as e:
    throw_exception(e)

# COMMAND ----------

try:
    # Fetching monitoring config
    monitoring_config = get_monitoring_config(deployment_master_id)
    if monitoring_config == []:
        declare_job_as_successful(no_monitor_config=True)
except Exception as e:
    throw_exception(e)

# COMMAND ----------

try:
    monitoring_activation_info = {
        "feature_drift": {"is_activated": "1", "start_date": "", "end_date": ""},
        "concept_drift": {"is_activated": "1", "start_date": "", "end_date": ""},
        "target_y_pred": {"is_activated": "1", "start_date": "", "end_date": ""},
        "target_y_actual": {"is_activated": "1", "start_date": "", "end_date": ""},
        "performance_drift": {"is_activated": "1", "start_date": "", "end_date": ""},
    }
    if "controls" in monitoring_config.keys():
        if len(monitoring_config["controls"]) > 0:
            monitoring_activation_info = monitoring_config["controls"][
                "monitor_control"
            ]
except Exception as e:
    throw_exception(e)

# COMMAND ----------

# DBTITLE 1,Get Model Artifact Details
model_details_json = get_model_details(model_artifact_id, h1)

model_name = model_details_json["model_name"]
model_version = model_details_json["model_version"]
model_session_id = model_details_json["model_train_session_id"]
modelling_task_type = model_details_json["model_train_variables"]["modelling_task_type"]
feature_columns = model_details_json["feature_columns"]
target_column = model_details_json["target_columns"]

if not isinstance(feature_columns, list):
    feature_columns = ast.literal_eval(feature_columns)
if not isinstance(target_column, list):
    target_column = ast.literal_eval(target_column)

prediction_column = ["prediction"]

try:
    partition_keys = ast.literal_eval(dbutils.widgets.get("partition_keys"))
except:
    partition_keys = []

# COMMAND ----------

# MAGIC %md
# MAGIC ## Get reference table

# COMMAND ----------

if datalake_env.lower() == "delta":

    monitoring_artifact_path = f"dbfs:/mnt/{az_container_name}/{env}/{project_id}/{version}/monitoring/{reference_table_id}/{current_table_id}/{model_artifact_id}"
else:
    monitoring_artifact_path = f"{env}_{project_id}_{version}_monitoring_{reference_table_id}_{current_table_id}_{model_artifact_id}"

if platform_datalake_env == "delta":
    task_log_path = f"dbfs:/mnt/{az_container_name}/{env}/{project_id}/{version}/{job_id}/task_log_table"
else:
    task_log_path = f"{env}_{project_id}_{version}_{job_id}_task_log_table"

# COMMAND ----------

table_details = {
    "model_name": model_name,
    "table_path": {
        "source_table_path": reference_table_path,
        "ground_truth_table_path": yactuals_table_path,
        "inference_task_log_path": inference_task_log_path,
        "inference_table_path": inference_table_path,
        "monitoring_task_log_path": task_log_path,
    },
    "feature_columns": feature_columns,
    "gt_column": target_column,
    "prediction_column": prediction_column,
    "primary_keys": primary_keys,
    "partition_keys": partition_keys,
}

utils.log(table_details, message_run)

# COMMAND ----------

print(table_details)

# COMMAND ----------

# try:
reference_table, current_table, inference_task_record = get_monitoring_tables(
    dbutils, spark, table_details
)
# except Exception as e:
#     traceback.print_exc()
#     # throw_exception(e)


# COMMAND ----------

#FIXME: REMOVE THIS
reference_table = reference_table.limit(1000)
current_table = current_table.limit(1000)

# COMMAND ----------

reference_table.display()

# COMMAND ----------

current_table.display()

# COMMAND ----------

inference_task_record

# COMMAND ----------

# MAGIC %md
# MAGIC ## Local Variables

# COMMAND ----------

utils.log(
    f"input params :\n\
    project_id     - {project_id},\n\
    version         - {version},\n\
    created_by_id   - {created_by_id},\n\
    datalake_env    - {datalake_env},\n\
    batch_size  - {batch_size},\n\
    mode -   {env},\n\
    model_artifact_id  - {model_artifact_id},\n\
    model_name      - {model_name},\n\
    model_version   - {model_version},\n\
    model_session_id   - {model_session_id},\n\
    modelling_task_type - {modelling_task_type},\n\
    reference_table_path  - {reference_table_path},\n\
    current_table_path       - {current_table_path},\n\
    task_log_path        - {task_log_path},\n\
    monitoring_artifact_path  - {monitoring_artifact_path},\n\
    reference_table    - {type(reference_table)},\n\
    feature_columns    - {feature_columns},\n\
    ",
    message_run,
)


aggregated_dd_table_name = "aggregated_monitoring_data_drift_table"
aggregated_pd_table_name = "aggregated_monitoring_performance_drift_table"
if platform_datalake_env.lower() == "delta":
    aggregated_performance_drift_path = (
        f"dbfs:/user/hive/warehouse/mlcore_observability_{deployment_env}.db/{aggregated_pd_table_name}"
    )
    aggregated_dd_drift_path = (
        f"dbfs:/user/hive/warehouse/mlcore_observability_{deployment_env}.db/{aggregated_dd_table_name}"
    )
else:
    aggregated_performance_drift_path = (
        f"{deployment_env}_monitoring_aggregated_performance_drift"
    )
    aggregated_dd_drift_path = f"{deployment_env}_monitoring_aggregated_data_drift"

# COMMAND ----------

# MAGIC %md
# MAGIC ## Monitoring Business Logic

# COMMAND ----------

# MAGIC %md
# MAGIC ### Model Monitoring Imports

# COMMAND ----------

from monitoring.utils.helpers import generate_monitoring_artifact_path

from monitoring.data.feature_drift.batch_monitors import (
    setup_feature_drift_monitors,
    monitor_feature_drift,
    monitor_feature_drift_in_memory,
)
from monitoring.data.target_drift.batch_monitors import (
    setup_target_drift_monitors,
    monitor_target_drift,
    monitor_target_drift_in_memory,
)
from monitoring.model.concept_drift.batch_monitors import (
    setup_concept_drift_monitors,
    monitor_concept_drift,
    monitor_concept_drift_in_memory,
)
from monitoring.model.performance_drift.batch_monitors import (
    setup_performance_drift_monitors,
    monitor_performance_drift,
    monitor_performance_drift_in_memory,
)

# COMMAND ----------

# MAGIC %md
# MAGIC ### Monitoring config
# MAGIC

# COMMAND ----------

monitoring_config

# COMMAND ----------

# MAGIC %md
# MAGIC ### Get Dataframes for each Monitoring subtype

# COMMAND ----------

# import traceback

# try:
# Fetching required properties
project_id = monitoring_config["project_id"]
version = monitoring_config["version"]
model_artifact_id = monitoring_config["model_artifact_id"]
input_table_id = monitoring_config["input_table_id"]
data_prep_deployment_job_id = monitoring_config["data_prep_deployment_job_id"]
actual_target = "y_actual"
predicted_target = "prediction"
null_gt_skipped_types = []

# Extracting monitoring data as per the configuration
if (
    "dq_drift" in monitoring_config.keys()
    and monitoring_activation_info.get("dq_drift", {}).get("is_activated", "0") == "1"
):
    dq_drift_conf = monitoring_config["dq_drift"]
else:
    dq_drift_conf = {
        "monitoring_type": "dq_drift",
        "monitoring_algo": [],
        "monitoring_mode": None,
    }

if (
    "feature_drift" in monitoring_config.keys()
    and monitoring_activation_info.get("feature_drift", {}).get("is_activated", "0")
    == "1"
):
    # Fetching feature drift configuration
    feature_drift_conf = monitoring_config["feature_drift"]
    reference_df = reference_table.toPandas().dropna(subset=feature_columns)

    # Fetching start date and end date from activation info
    start_date = monitoring_activation_info["concept_drift"]["start_date"]
    end_date = monitoring_activation_info["concept_drift"]["end_date"]

    # Fetching next batch of Inference data
    current_df, current_df_inference_entry = current_table, inference_task_record
    try:
        current_df = current_df.toPandas()
    except:
        utils.log(f"Current Df was sent as None, batches are exhausted.", message_run)
        feature_drift_conf = {
            "monitoring_type": "feature_drift",
            "monitoring_algo_overall": [],
            "monitoring_algo_features": [],
            "monitoring_mode": None,
        }
        reference_df = None
        current_df = None
else:
    utils.log(
        "Either feature_drift key is not there in config OR is_activated is 0",
        message_run,
    )
    feature_drift_conf = {
        "monitoring_type": "feature_drift",
        "monitoring_algo_overall": [],
        "monitoring_algo_features": [],
        "monitoring_mode": None,
    }
    reference_df = None
    current_df = None

if "target_drift" in monitoring_config.keys() and (
    monitoring_activation_info.get("target_y_pred", {}).get("is_activated", "0") == "1"
    or monitoring_activation_info.get("target_y_actual", {}).get("is_activated", "0")
    == "1"
):
    # Fetching target drift configuration
    target_drift_conf = monitoring_config["target_drift"]
    actual_target = "y_actual"
    predicted_target = "prediction"

    # Fetching start date and end date from activation info
    start_date = monitoring_activation_info["concept_drift"]["start_date"]
    end_date = monitoring_activation_info["concept_drift"]["end_date"]

    # Getting y_pred for reference table
    tdp_reference_df = reference_table.toPandas().dropna(subset=["prediction"])

    # Getting inference data joined batch for target y pred
    tdp_current_df, tdp_current_df_inference_entry = (
        current_table,
        inference_task_record,
    )
    try:
        tdp_current_df = tdp_current_df.toPandas()
    except:
        utils.log(
            f"tdp_current_df was sent as None, batches are exhausted.", message_run
        )
        tdp_current_df = None

    # Getting y_true for reference table
    tda_reference_df = reference_table.toPandas().dropna(subset=["y_actual"])

    # Getting inference data joined batch for target y actual
    tda_current_df, tda_current_df_inference_entry = (
        current_table,
        inference_task_record,
    )

    try:
        # If null values exist in actual target column, we skip the execution to maintain inference batch lineage
        y_actual_null_count = tda_current_df.select(
            [
                F.count(F.when(F.isnan(c) | F.col(c).isNull(), c)).alias(c)
                for c in [actual_target]
            ]
        ).first()[actual_target]

        if y_actual_null_count > 0:
            utils.log(
                "TARGET_DRIFT: If null values exist in actual target column, we skip the execution to maintain inference batch lineage",
                message_run,
            )
            target_drift_conf = {
                "monitoring_type": "target_drift",
                "monitoring_algo": [],
                "monitoring_mode": None,
            }
            tda_reference_df = None
            tda_current_df = None
            tdp_reference_df = None
            tdp_current_df = None
            null_gt_skipped_types.append("target_y_actual")
        else:
            tda_current_df = tda_current_df.toPandas()
    except:
        utils.log("Target Drift Data Fetch Exception", message_run)
        traceback.print_exc()
        target_drift_conf = {
            "monitoring_type": "target_drift",
            "monitoring_algo": [],
            "monitoring_mode": None,
        }
        tda_reference_df = None
        tda_current_df = None
        tdp_reference_df = None
        tdp_current_df = None

else:
    utils.log(
        "Either target_drift key is not there in config OR is_activated is 0",
        message_run,
    )
    target_drift_conf = {
        "monitoring_type": "target_drift",
        "monitoring_algo": [],
        "monitoring_mode": None,
    }
    tda_reference_df = None
    tda_current_df = None
    tdp_reference_df = None
    tdp_current_df = None

if (
    "concept_drift" in monitoring_config.keys()
    and monitoring_activation_info.get("concept_drift", {}).get("is_activated", "0")
    == "1"
):
    # Fetching concept drift configuration
    concept_drift_conf = monitoring_config["concept_drift"]
    actual_target = "y_actual"
    predicted_target = "prediction"

    # Fetching start date and end date from activation info
    start_date = monitoring_activation_info["concept_drift"]["start_date"]
    end_date = monitoring_activation_info["concept_drift"]["end_date"]

    # Getting y_true and y_pred for reference table
    ref_df_w_yact_ypred = (
        reference_table.dropna().toPandas()
    )

    # Fetching data required for concept drift
    curr_df_w_yact_ypred_conc = current_table
    curr_df_w_yact_ypred_con_inference_entry = inference_task_record

    try:
        # If null values exist in actual target column, we skip the execution to maintain inference batch lineage
        y_actual_null_count = curr_df_w_yact_ypred_conc.select(
            [
                F.count(F.when(F.isnan(c) | F.col(c).isNull(), c)).alias(c)
                for c in [actual_target]
            ]
        ).first()[actual_target]
        if y_actual_null_count > 0:
            utils.log(
                "CONCEPT_DRIFT: If null values exist in actual target column, we skip the execution to maintain inference batch lineage",
                message_run,
            )
            concept_drift_conf = {
                "monitoring_type": "concept_drift",
                "monitoring_algo": [],
                "monitoring_mode": None,
            }
            cda_reference_df = None
            cdp_reference_df = None
            cda_current_df = None
            cdp_current_df = None
            null_gt_skipped_types.append("concept_drift")
        else:
            curr_df_w_yact_ypred_conc = curr_df_w_yact_ypred_conc.toPandas()
            cda_reference_df = ref_df_w_yact_ypred
            cdp_reference_df = ref_df_w_yact_ypred
            cda_current_df = curr_df_w_yact_ypred_conc
            cdp_current_df = curr_df_w_yact_ypred_conc
    except:
        utils.log("Concept Drift Data Fetch Exception", message_run)
        traceback.print_exc()
        concept_drift_conf = {
            "monitoring_type": "concept_drift",
            "monitoring_algo": [],
            "monitoring_mode": None,
        }
        cda_reference_df = None
        cdp_reference_df = None
        cda_current_df = None
        cdp_current_df = None
else:
    utils.log(
        "Either concept_drift key is not there in config OR is_activated is 0",
        message_run,
    )
    concept_drift_conf = {
        "monitoring_type": "concept_drift",
        "monitoring_algo": [],
        "monitoring_mode": None,
    }
    cda_reference_df = None
    cdp_reference_df = None
    cda_current_df = None
    cdp_current_df = None

if (
    "performance_drift" in monitoring_config.keys()
    and monitoring_activation_info.get("performance_drift", {}).get("is_activated", "0")
    == "1"
):
    # Fetching performance drift configuration
    performance_drift_conf = monitoring_config["performance_drift"]
    for metric_dict in performance_drift_conf["monitoring_metrics"]:
        metric_dict["algo_name"] = metric_dict["metrics"]
    actual_target = "y_actual"
    predicted_target = "prediction"

    # Fetching start date and end date
    start_date = monitoring_activation_info["performance_drift"]["start_date"]
    end_date = monitoring_activation_info["performance_drift"]["end_date"]

    # Getting y_true and y_pred for reference table
    ref_df_w_yact_ypred = (
        reference_table.dropna().toPandas()
    )

    # Fetching data required for performance drift
    curr_df_w_yact_ypred_perf = current_table
    curr_df_w_yact_ypred_perf_inference_entry = inference_task_record
    try:
        # If null values exist in actual target column, we skip the execution to maintain inference batch lineage
        y_actual_null_count = curr_df_w_yact_ypred_perf.select(
            [
                F.count(F.when(F.isnan(c) | F.col(c).isNull(), c)).alias(c)
                for c in [actual_target]
            ]
        ).first()[actual_target]
        if y_actual_null_count > 0:
            utils.log(
                "PERF_DRIFT: If null values exist in actual target column, we skip the execution to maintain inference batch lineage",
                message_run,
            )
            performance_drift_conf = {
                "monitoring_type": "performance_drift",
                "monitoring_metrics": [],
                "monitoring_mode": None,
            }
            pda_reference_df = None
            pdp_reference_df = None
            pda_current_df = None
            pdp_current_df = None
            null_gt_skipped_types.append("performance_drift")
        else:
            curr_df_w_yact_ypred_perf = curr_df_w_yact_ypred_perf.toPandas()
            pda_reference_df = ref_df_w_yact_ypred
            pdp_reference_df = ref_df_w_yact_ypred
            pda_current_df = curr_df_w_yact_ypred_perf
            pdp_current_df = curr_df_w_yact_ypred_perf
    except:
        utils.log("Performance drift Data Fetch Exception", message_run)
        traceback.print_exc()
        performance_drift_conf = {
            "monitoring_type": "performance_drift",
            "monitoring_metrics": [],
            "monitoring_mode": None,
        }
        pda_reference_df = None
        pdp_reference_df = None
        pda_current_df = None
        pdp_current_df = None
else:
    utils.log(
        "Either performance_drift key is not there in config OR is_activated is 0",
        message_run,
    )
    performance_drift_conf = {
        "monitoring_type": "performance_drift",
        "monitoring_metrics": [],
        "monitoring_mode": None,
    }
    pda_reference_df = None
    pdp_reference_df = None
    pda_current_df = None
    pdp_current_df = None

# except Exception as e:
# throw_exception(e)

# COMMAND ----------

# MAGIC %md
# MAGIC ### Monitoring Metric JSON Template

# COMMAND ----------

monitoring_metrics_template_json = {
    "model_artifact_id": model_artifact_id,
    "input_table_id": input_table_id,
    "data_prep_deployment_job_id": data_prep_deployment_job_id,
    "deployment_master_id": deployment_master_id,
    "processing_time": int(time.time() * 1000000),
    "batch_start_id": "1",
    "batch_end_id": "10000",
    "batch_size": batch_size,
    "monitoring_type": None,
    "monitoring_sub_type": None,
    "monitoring_algo": None,
    "env": env,
    "data": None,
    "created_by_name": created_by_name,
    "created_by_id": created_by_id,
    "model_name": model_name,
    "model_version": model_version,
    "model_session_id": model_session_id,
    "project_id": project_id,
    "project_name": project_name,
    "version": version,
    "inference_metadata": str(current_table.toPandas().describe().to_dict("records")),
}

# COMMAND ----------

# MAGIC %md
# MAGIC ### FEATURE DRIFT

# COMMAND ----------

# MAGIC %md
# MAGIC #### Feature Drift Inputs
# MAGIC

# COMMAND ----------

# MAGIC %md
# MAGIC MODIFIED BELOW

# COMMAND ----------

granularity_levels = monitoring_config['granularity']['monitor_granularity']['granularity_level']
print(granularity_levels)

# COMMAND ----------

def get_grouped_df_dict(granularity_levels,reference_df,current_df):
    # Create a dictionary to store the DataFrames for reference_table and current_table
    df_dict = {
        'reference_tables': {},
        'current_tables': {}
    }
    reference_grouped = reference_df.groupby(list(granularity_levels.keys()))
    current_grouped = current_df.groupby(list(granularity_levels.keys()))
    # Create a dictionary to hold the separate DataFrames
    df_dict["reference_tables"] = {category: group for category, group in reference_grouped}
    df_dict["current_tables"] = {category: group for category, group in current_grouped}

    #Keep common keys
    common_keys = set(df_dict["reference_tables"].keys()) & set(df_dict["current_tables"].keys())
    df_dict["reference_tables"] = {key: df_dict["reference_tables"][key] for key in common_keys}
    df_dict["current_tables"] = {key: df_dict["current_tables"][key] for key in common_keys}

    return df_dict

# COMMAND ----------

df_dict = get_grouped_df_dict(granularity_levels,reference_df,current_df)

# COMMAND ----------

df_dict['reference_tables'].keys()

# COMMAND ----------

df_dict['current_tables'].keys()

# COMMAND ----------

def data_type_detection(feature_drift_conf,target_drift_conf):
    data_type = "tabular"
    try:
        for algo_dict in feature_drift_conf["monitoring_algo_overall"]:
            for key in algo_dict.keys():
                if algo_dict[key] in ["DD3", "DD4", "DD5", "FD3", "FD4", "FD5", "FD14"]:
                    data_type = "timeseries"
        for algo_dict in target_drift_conf["monitoring_algo"]:
            for key in algo_dict.keys():
                if algo_dict[key] in ["DD3", "DD4", "DD5", "TD12", "TD13", "TD14", "TD15"]:
                    data_type = "timeseries"
        return data_type
    except Exception as e:
        utils.log(str(e), message_run)
        return data_type

# COMMAND ----------

# FIXME: Random picks 1 date column for TS, need a better way to get datecol
def remove_non_num_cols(reference_table,feature_drift_conf,feature_columns,data_type):
    date_column = None
    feature_columns = list_numerical_columns(spark.createDataFrame(reference_table[feature_columns]))
    feature_drift_conf["monitored_features"] = feature_columns


    if data_type == "timeseries":
        reference_features = ast.literal_eval(feature_drift_conf["reference_features"])
        date_column_list = list_datelike_columns(spark.createDataFrame(reference_table[reference_features]))
        if date_column_list:
            date_column = date_column_list[0]
        else:
            utils.log(
                "TIMESERIES data detected, but no date column found in reference_features!",
                message_run,
            )
    utils.log(f"monitored_features {feature_columns}", message_run)
    utils.log(f"date_column {date_column}", message_run)
    return feature_drift_conf,date_column

# COMMAND ----------

def fd_in_memory(feature_drift_conf,fd_reference_df,fd_current_df,listed_feature_monitors,date_column,data_type):
    try:
        feature_algo_thresholds = (
            feature_drift_conf["monitoring_algo_feature_thresholds"]
            + feature_drift_conf["monitoring_algo_overall_thresholds"]
        )

        monitored_features = feature_drift_conf["monitored_features"]

        # Sample size selection
        DATA_DRIFT_SAMPLE = 5000

        if fd_reference_df.shape[0] < DATA_DRIFT_SAMPLE:
            REF_DATA_DRIFT_SAMPLE = fd_reference_df.shape[0]
        else:
            REF_DATA_DRIFT_SAMPLE = DATA_DRIFT_SAMPLE
        if fd_current_df.shape[0] < DATA_DRIFT_SAMPLE:
            CURR_DATA_DRIFT_SAMPLE = fd_current_df.shape[0]
        else:
            CURR_DATA_DRIFT_SAMPLE = DATA_DRIFT_SAMPLE

        feature_drift_metrics_list = monitor_feature_drift_in_memory(
            data_type=data_type,
            reference_df=fd_reference_df.sample(n=REF_DATA_DRIFT_SAMPLE),
            current_df=fd_current_df.sample(n=CURR_DATA_DRIFT_SAMPLE),
            monitored_features=monitored_features,
            monitoring_algos=listed_feature_monitors,
            monitoring_algo_thresholds=feature_algo_thresholds,
            date_column=date_column,
        )
    except Exception as e:
        utils.log(str(e), message_run)
        traceback.print_exc()
        feature_drift_metrics_list = []
    return feature_drift_metrics_list

# COMMAND ----------

from monitoring.algorithms.batch_algorithms import fdr_correction

def apply_fdr_correction(feature_drift_conf,feature_drift_metrics_list):
    # TODO: Take Dynamic Threshold
    valid_overall_algos = map_algo_id_2_name(
        [
            algo_dict["algo_id"]
            for algo_dict in feature_drift_conf["monitoring_algo_overall"]
        ]
    )
    feature_drift_overall_metrics_list = []
    to_be_removed_index = []
    for i, drift_json in enumerate(feature_drift_metrics_list):
        if drift_json["monitor_algo"] in valid_overall_algos:
            fdr_drift_json = fdr_correction(
                monitor_algo=drift_json["monitor_algo"],
                p_values=drift_json["p_val"],
                threshold=0.05,
            )
            utils.log(fdr_drift_json, message_run)
            fdr_drift_json["p_val"] = max(fdr_drift_json["p_val"])
            feature_drift_overall_metrics_list.append(fdr_drift_json)
            to_be_removed_index.append(i)
    utils.log(to_be_removed_index, message_run)
    feature_drift_metrics_list = [
        e for i, e in enumerate(feature_drift_metrics_list) if i not in to_be_removed_index
    ]

    return feature_drift_metrics_list

# COMMAND ----------

def create_fd_metric_json_list(feature_drift_metrics_list,segment):
    feature_drift_metric_json_list = []
    i = 0
    j = 0
    for metric in feature_drift_metrics_list:
        feature_drift_metric_json = monitoring_metrics_template_json.copy()
        feature_drift_metric_json["granularity"] = segment
        feature_drift_metric_json["monitoring_type"] = "feature_drift"
        if "list" in str(type(metric["is_drift"])):
            # Feature Level Drift Metric
            feature_drift_metric_json["monitoring_sub_type"] = "feature_level"
            feature_drift_metric_json["monitoring_algo"] = feature_drift_conf[
                "monitoring_algo_features"
            ][i]
            metric.pop("monitor_algo", None)
            feature_drift_metric_json["data"] = round_metrics(metric)
            print(f"FEATURE_DRIFT ======================== {feature_drift_metric_json}")
            utils.log("FEATURE_DRIFT Calculation completed", message_run)
            i = i + 1
        else:
            # Overall Drift Metric
            feature_drift_metric_json["monitoring_sub_type"] = "overall"
            feature_drift_metric_json["monitoring_algo"] = feature_drift_conf[
                "monitoring_algo_overall"
            ][j]
            metric.pop("monitor_algo", None)
            feature_drift_metric_json["data"] = round_metrics(metric)
            print(f"OVERALL_DRIFT ======================== {feature_drift_metric_json}")
            utils.log("OVERALL_DRIFT Calculation completed", message_run)
            j = j + 1
        feature_drift_metric_json_list.append(feature_drift_metric_json)
    return feature_drift_metric_json_list

# COMMAND ----------

def fd_in_memory_segmented(df_dict,feature_drift_conf,target_drift_conf):
    FINAL_FD_LIST = []
    data_type =  data_type_detection(feature_drift_conf,target_drift_conf)
    feature_monitor_algo_to_setup, listed_feature_monitors = check_for_monitor_setup(drift_conf=feature_drift_conf, target_type=None)
    for k,fd_reference_df in df_dict['reference_tables'].items():
        feature_drift_conf,date_column = remove_non_num_cols(fd_reference_df,feature_drift_conf,feature_columns,data_type)
        feature_drift_metrics_list = fd_in_memory(feature_drift_conf,fd_reference_df,df_dict['current_tables'][k],listed_feature_monitors,date_column,data_type)
        feature_drift_metrics_list = apply_fdr_correction(feature_drift_conf,feature_drift_metrics_list)
        feature_drift_metric_json_list = create_fd_metric_json_list(feature_drift_metrics_list,str(k))
        FINAL_FD_LIST.extend(feature_drift_metric_json_list)
    
    return FINAL_FD_LIST

# COMMAND ----------

feature_drift_metric_json_list = fd_in_memory_segmented(df_dict,feature_drift_conf,target_drift_conf)

# COMMAND ----------

len(feature_drift_metric_json_list)

# COMMAND ----------

# MAGIC %md
# MAGIC MODIFIED ABOVE

# COMMAND ----------

# fd_reference_df = reference_df
# fd_current_df = current_df

# COMMAND ----------

# fd_reference_df

# COMMAND ----------

# fd_current_df

# COMMAND ----------

# feature_drift_conf

# COMMAND ----------

# MAGIC %md
# MAGIC #### Check if Artifacts have been setup for Feature Drift Config

# COMMAND ----------

# feature_monitor_algo_to_setup, listed_feature_monitors = check_for_monitor_setup(
#     drift_conf=feature_drift_conf, target_type=None
# )

# COMMAND ----------

# MAGIC %md
# MAGIC #### Setup Feature Drift Monitoring Artifacts if needed

# COMMAND ----------

# FIXME: Temp detection of data_type
data_type = "tabular"
try:
    for algo_dict in feature_drift_conf["monitoring_algo_overall"]:
        for key in algo_dict.keys():
            if algo_dict[key] in ["DD3", "DD4", "DD5", "FD3", "FD4", "FD5", "FD14"]:
                data_type = "timeseries"
    for algo_dict in target_drift_conf["monitoring_algo"]:
        for key in algo_dict.keys():
            if algo_dict[key] in ["DD3", "DD4", "DD5", "TD12", "TD13", "TD14", "TD15"]:
                data_type = "timeseries"

except Exception as e:
    utils.log(str(e), message_run)

# Setup Monitoring Artifacts
# setup_monitor(
#     monitor_algo_to_setup=feature_monitor_algo_to_setup,
#     drift_conf=feature_drift_conf,
#     data_type=data_type,
#     reference_df=fd_reference_df,
#     monitoring_artifact_path=monitoring_artifact_path,
# )

# COMMAND ----------

# MAGIC %md
# MAGIC #### Remove non num features from feature drift monitoring

# COMMAND ----------

# FIXME: Random picks 1 date column for TS, need a better way to get datecol
date_column = None
feature_columns = list_numerical_columns(reference_table.select(*feature_columns))
feature_drift_conf["monitored_features"] = feature_columns


if data_type == "timeseries":
    reference_features = ast.literal_eval(feature_drift_conf["reference_features"])
    date_column_list = list_datelike_columns(
        reference_table.select(*reference_features)
    )
    if date_column_list:
        date_column = date_column_list[0]
    else:
        utils.log(
            "TIMESERIES data detected, but no date column found in reference_features!",
            message_run,
        )
utils.log(f"monitored_features {feature_columns}", message_run)
utils.log(f"date_column {date_column}", message_run)

# COMMAND ----------

# MAGIC %md
# MAGIC #### Monitor Feature Drift for Current DF

# COMMAND ----------

# feature_drift_metrics_list = monitor_current_data(
#     drift_conf=feature_drift_conf,
#     data_type=data_type,
#     current_df=fd_current_df,
#     monitoring_algos=listed_feature_monitors,
#     monitoring_artifact_path=monitoring_artifact_path,
# )

# COMMAND ----------

# try:
#     feature_algo_thresholds = (
#         feature_drift_conf["monitoring_algo_feature_thresholds"]
#         + feature_drift_conf["monitoring_algo_overall_thresholds"]
#     )

#     monitored_features = feature_drift_conf["monitored_features"]

#     # Sample size selection
#     DATA_DRIFT_SAMPLE = 5000

#     if fd_reference_df.shape[0] < DATA_DRIFT_SAMPLE:
#         REF_DATA_DRIFT_SAMPLE = fd_reference_df.shape[0]
#     else:
#         REF_DATA_DRIFT_SAMPLE = DATA_DRIFT_SAMPLE
#     if fd_current_df.shape[0] < DATA_DRIFT_SAMPLE:
#         CURR_DATA_DRIFT_SAMPLE = fd_current_df.shape[0]
#     else:
#         CURR_DATA_DRIFT_SAMPLE = DATA_DRIFT_SAMPLE

#     feature_drift_metrics_list = monitor_feature_drift_in_memory(
#         data_type=data_type,
#         reference_df=fd_reference_df.sample(n=REF_DATA_DRIFT_SAMPLE),
#         current_df=fd_current_df.sample(n=CURR_DATA_DRIFT_SAMPLE),
#         monitored_features=monitored_features,
#         monitoring_algos=listed_feature_monitors,
#         monitoring_algo_thresholds=feature_algo_thresholds,
#         date_column=date_column,
#     )
# except Exception as e:
#     utils.log(str(e), message_run)
#     traceback.print_exc()
#     feature_drift_metrics_list = []

# COMMAND ----------

# feature_drift_metrics_list

# COMMAND ----------

# MAGIC %md
# MAGIC #### Apply FDR correction for DD algos

# COMMAND ----------

# from monitoring.algorithms.batch_algorithms import fdr_correction

# # TODO: Take Dynamic Threshold
# valid_overall_algos = map_algo_id_2_name(
#     [
#         algo_dict["algo_id"]
#         for algo_dict in feature_drift_conf["monitoring_algo_overall"]
#     ]
# )
# feature_drift_overall_metrics_list = []
# to_be_removed_index = []
# for i, drift_json in enumerate(feature_drift_metrics_list):
#     if drift_json["monitor_algo"] in valid_overall_algos:
#         fdr_drift_json = fdr_correction(
#             monitor_algo=drift_json["monitor_algo"],
#             p_values=drift_json["p_val"],
#             threshold=0.05,
#         )
#         utils.log(fdr_drift_json, message_run)
#         fdr_drift_json["p_val"] = max(fdr_drift_json["p_val"])
#         feature_drift_overall_metrics_list.append(fdr_drift_json)
#         to_be_removed_index.append(i)
# utils.log(to_be_removed_index, message_run)
# feature_drift_metrics_list = [
#     e for i, e in enumerate(feature_drift_metrics_list) if i not in to_be_removed_index
# ]

# COMMAND ----------

# feature_drift_metrics_list = (
#     feature_drift_metrics_list + feature_drift_overall_metrics_list
# )
# utils.log(
#     f"feature_drift_metrics_list =====> : {feature_drift_metrics_list}", message_run
# )

# COMMAND ----------

# MAGIC %md
# MAGIC #### Calculate Feature Drift Metrics JSON List

# COMMAND ----------

# feature_drift_metric_json_list = []
# i = 0
# j = 0
# for metric in feature_drift_metrics_list:
#     feature_drift_metric_json = monitoring_metrics_template_json.copy()
#     feature_drift_metric_json["monitoring_type"] = "feature_drift"
#     if "list" in str(type(metric["is_drift"])):
#         # Feature Level Drift Metric
#         feature_drift_metric_json["monitoring_sub_type"] = "feature_level"
#         feature_drift_metric_json["monitoring_algo"] = feature_drift_conf[
#             "monitoring_algo_features"
#         ][i]
#         metric.pop("monitor_algo", None)
#         feature_drift_metric_json["data"] = round_metrics(metric)
#         print(f"FEATURE_DRIFT ======================== {feature_drift_metric_json}")
#         utils.log("FEATURE_DRIFT Calculation completed", message_run)
#         i = i + 1
#     else:
#         # Overall Drift Metric
#         feature_drift_metric_json["monitoring_sub_type"] = "overall"
#         feature_drift_metric_json["monitoring_algo"] = feature_drift_conf[
#             "monitoring_algo_overall"
#         ][j]
#         metric.pop("monitor_algo", None)
#         feature_drift_metric_json["data"] = round_metrics(metric)
#         print(f"OVERALL_DRIFT ======================== {feature_drift_metric_json}")
#         utils.log("OVERALL_DRIFT Calculation completed", message_run)
#         j = j + 1
#     feature_drift_metric_json_list.append(feature_drift_metric_json)

# COMMAND ----------

# MAGIC %md
# MAGIC ### TARGET DRIFT

# COMMAND ----------

# MAGIC %md
# MAGIC #### Target Drift Inputs
# MAGIC

# COMMAND ----------

# MAGIC %md
# MAGIC MODIFIED BELOW

# COMMAND ----------

df_dict_tda = get_grouped_df_dict(granularity_levels,tda_reference_df,tda_current_df)

# COMMAND ----------

df_dict_tda['reference_tables'].keys()

# COMMAND ----------

df_dict_tda['current_tables'].keys()

# COMMAND ----------

def td_in_memory(target_drift_conf,td_reference_df,td_current_df,listed_target_monitors,date_column,data_type,target,target_type):
    target_drift_metrics_list = monitor_target_drift_in_memory(
    data_type=data_type,
    reference_df=td_reference_df,
    current_df=td_current_df,
    target=target,
    target_type=target_type,
    monitoring_algo_thresholds=None,
    monitoring_algos=listed_target_monitors,
    date_column=date_column)
    return target_drift_metrics_list

# COMMAND ----------

def create_td_metric_json_list(target_drift_metrics_list,target_type,segment):
    target_drift_metric_json_list = []

    for i, metric in enumerate(target_drift_metrics_list):
        target_drift_metric_json = monitoring_metrics_template_json.copy()
        target_drift_metric_json["granularity"] = segment
        target_drift_metric_json["monitoring_type"] = "target_drift"
        # Actual Target Drift Metric
        target_drift_metric_json["monitoring_sub_type"] = target_type
        target_drift_metric_json["monitoring_algo"] = target_drift_conf["monitoring_algo"][
            i
        ]
        metric.pop("monitor_algo", None)
        target_drift_metric_json["data"] = round_metrics(metric)
        print(f"{target_type} TARGET DRIFT========================> {target_drift_metric_json}")
        utils.log("{target_type} TARGET DRIFT Calculation completed", message_run)
        target_drift_metric_json_list.append(target_drift_metric_json)
    return target_drift_metric_json_list

# COMMAND ----------

def tda_in_memory_segmented(df_dict_tda,feature_drift_conf,target_drift_conf):
    FINAL_TDA_LIST = []
    data_type =  data_type_detection(feature_drift_conf,target_drift_conf)
    actual_target_monitor_algo_to_setup, listed_target_monitors = check_for_monitor_setup(target_drift_conf, target_type="actual")
    for k,tda_reference_df in df_dict_tda['reference_tables'].items():
        feature_drift_conf,date_column = remove_non_num_cols(tda_reference_df,feature_drift_conf,feature_columns,data_type)
        actual_target_drift_metrics_list = td_in_memory(target_drift_conf,tda_reference_df,df_dict_tda['current_tables'][k],listed_target_monitors,date_column,data_type,actual_target,"actual")
        actual_target_drift_metric_json_list = create_td_metric_json_list(actual_target_drift_metrics_list,"actual",str(k))
        FINAL_TDA_LIST.extend(actual_target_drift_metric_json_list)
    return FINAL_TDA_LIST

# COMMAND ----------

actual_target_drift_metric_json_list = tda_in_memory_segmented(df_dict_tda,feature_drift_conf,target_drift_conf)

# COMMAND ----------

len(actual_target_drift_metric_json_list)

# COMMAND ----------

df_dict_tdp =  get_grouped_df_dict(granularity_levels,tdp_reference_df,tdp_current_df)

# COMMAND ----------

def tdp_in_memory_segmented(df_dict_tdp,feature_drift_conf,target_drift_conf):
    FINAL_TDP_LIST = []
    data_type =  data_type_detection(feature_drift_conf,target_drift_conf)
    predicted_target_monitor_algo_to_setup, listed_target_monitors = check_for_monitor_setup(target_drift_conf, target_type="actual")
    for k,tdp_reference_df in df_dict_tdp['reference_tables'].items():
        feature_drift_conf,date_column = remove_non_num_cols(tda_reference_df,feature_drift_conf,feature_columns,data_type)
        predicted_target_drift_metrics_list = td_in_memory(target_drift_conf,tdp_reference_df,df_dict_tdp['current_tables'][k],listed_target_monitors,date_column,data_type,predicted_target,"predicted")
        predicted_target_drift_metric_json_list = create_td_metric_json_list(predicted_target_drift_metrics_list,"predicted",str(k))
        FINAL_TDP_LIST.extend(predicted_target_drift_metric_json_list)
    return FINAL_TDP_LIST

# COMMAND ----------

predicted_target_drift_metric_json_list = tdp_in_memory_segmented(df_dict_tdp,feature_drift_conf,target_drift_conf)

# COMMAND ----------

len(predicted_target_drift_metric_json_list)

# COMMAND ----------

target_drift_metric_json_list = (
    actual_target_drift_metric_json_list + predicted_target_drift_metric_json_list
)

# COMMAND ----------

# MAGIC %md
# MAGIC MODIFIED ABOVE

# COMMAND ----------

# tda_reference_df

# COMMAND ----------

# tda_current_df

# COMMAND ----------

# tdp_reference_df

# COMMAND ----------

# tdp_current_df

# COMMAND ----------

# target_drift_conf

# COMMAND ----------

# MAGIC %md
# MAGIC #### Check if Artifacts have been setup for Actual Target Drift Config

# COMMAND ----------

# actual_target_monitor_algo_to_setup, listed_target_monitors = check_for_monitor_setup(
#     target_drift_conf, target_type="actual"
# )

# COMMAND ----------

# MAGIC %md
# MAGIC #### Setup Actual Target Drift Monitoring Artifacts if needed

# COMMAND ----------

# data_type = "tabular"
# try:
#     for algo_dict in target_drift_conf["monitoring_algo"]:
#         for key in algo_dict.keys():
#             if algo_dict[key] in ["DD3", "DD4", "DD5", "TD12", "TD13", "TD14", "TD15"]:
#                 data_type = "timeseries"
# except Exception as e:
#     utils.log(str(e), message_run)

# # Setup Monitoring Artifacts
# setup_monitor(
#     monitor_algo_to_setup=actual_target_monitor_algo_to_setup,
#     drift_conf=target_drift_conf,
#     data_type=data_type,
#     reference_df=tda_reference_df,
#     monitoring_artifact_path=monitoring_artifact_path,
#     target=actual_target,
#     target_type="actual",
#     date_column=date_column,
# )

# COMMAND ----------

# MAGIC %md
# MAGIC #### Monitor Actual Target Drift for Current DF

# COMMAND ----------

# actual_target_drift_metrics_list = monitor_current_data(
#     drift_conf=target_drift_conf,
#     data_type=data_type,
#     current_df=tda_current_df,
#     target=actual_target,
#     target_type="actual",
#     monitoring_algos=listed_target_monitors,
#     monitoring_artifact_path=monitoring_artifact_path,
#     date_column=date_column,
# )

# COMMAND ----------

# actual_target_drift_metrics_list

# COMMAND ----------

# MAGIC %md
# MAGIC #### Check if Artifacts have been setup for Predicted Target Drift Config

# COMMAND ----------

# (
#     predicted_target_monitor_algo_to_setup,
#     listed_target_monitors,
# ) = check_for_monitor_setup(target_drift_conf, target_type="predicted")

# COMMAND ----------

# MAGIC %md
# MAGIC #### Setup Predicted Target Drift Monitoring Artifacts if needed

# COMMAND ----------

# # Setup Monitoring Artifacts
# setup_monitor(
#     monitor_algo_to_setup=actual_target_monitor_algo_to_setup,
#     drift_conf=target_drift_conf,
#     data_type=data_type,
#     reference_df=tdp_reference_df,
#     monitoring_artifact_path=monitoring_artifact_path,
#     target=predicted_target,
#     target_type="predicted",
#     date_column=date_column,
# )

# COMMAND ----------

# MAGIC %md
# MAGIC #### Monitor Predicted Target Drift for Current DF

# COMMAND ----------

# predicted_target_drift_metrics_list = monitor_current_data(
#     drift_conf=target_drift_conf,
#     data_type=data_type,
#     current_df=tdp_current_df,
#     target=predicted_target,
#     target_type="predicted",
#     monitoring_algos=listed_target_monitors,
#     monitoring_artifact_path=monitoring_artifact_path,
#     date_column=date_column,
# )

# COMMAND ----------

# predicted_target_drift_metrics_list

# COMMAND ----------

# MAGIC %md
# MAGIC #### Calculate Target Drift Metrics JSON List

# COMMAND ----------

# actual_target_drift_metric_json_list = []

# for i, metric in enumerate(actual_target_drift_metrics_list):
#     target_drift_metric_json = monitoring_metrics_template_json.copy()
#     target_drift_metric_json["monitoring_type"] = "target_drift"
#     # Actual Target Drift Metric
#     target_drift_metric_json["monitoring_sub_type"] = "actual"
#     target_drift_metric_json["monitoring_algo"] = target_drift_conf["monitoring_algo"][
#         i
#     ]
#     metric.pop("monitor_algo", None)
#     target_drift_metric_json["data"] = round_metrics(metric)
#     print(f"ACTUAL_DRIFT ========================> {target_drift_metric_json}")
#     utils.log("ACTUAL_DRIFT Calculation completed", message_run)
#     actual_target_drift_metric_json_list.append(target_drift_metric_json)

# predicted_target_drift_metric_json_list = []
# for i, metric in enumerate(predicted_target_drift_metrics_list):
#     target_drift_metric_json = monitoring_metrics_template_json.copy()
#     target_drift_metric_json["monitoring_type"] = "target_drift"
#     # Predicted Target Drift Metric
#     target_drift_metric_json["monitoring_sub_type"] = "predicted"
#     target_drift_metric_json["monitoring_algo"] = target_drift_conf["monitoring_algo"][
#         i
#     ]
#     metric.pop("monitor_algo", None)
#     target_drift_metric_json["data"] = round_metrics(metric)
#     print(f"PREDICTED_DRIFT ========================> {target_drift_metric_json}")
#     utils.log("PREDICTED_DRIFT Calculation completed", message_run)
#     predicted_target_drift_metric_json_list.append(target_drift_metric_json)

# target_drift_metric_json_list = (
#     actual_target_drift_metric_json_list + predicted_target_drift_metric_json_list
# )

# COMMAND ----------

# MAGIC %md
# MAGIC ### CONCEPT DRIFT

# COMMAND ----------

# MAGIC %md
# MAGIC #### Concept Drift Inputs
# MAGIC

# COMMAND ----------

# MAGIC %md
# MAGIC MODIFIED BELOW

# COMMAND ----------

df_dict_cda = get_grouped_df_dict(granularity_levels,cda_reference_df,cda_current_df)
df_dict_cdp = get_grouped_df_dict(granularity_levels,cdp_reference_df,cdp_current_df)

# COMMAND ----------

 #Keep common keys
common_keys = set(df_dict_cda["reference_tables"].keys()) & set(df_dict_cdp["reference_tables"].keys())
df_dict_cda["reference_tables"] = {key: df_dict_cda["reference_tables"][key] for key in common_keys}
df_dict_cdp["reference_tables"] = {key: df_dict_cdp["reference_tables"][key] for key in common_keys}

#Keep common keys
common_keys = set(df_dict_cda["current_tables"].keys()) & set(df_dict_cdp["current_tables"].keys())
df_dict_cda["current_tables"] = {key: df_dict_cda["current_tables"][key] for key in common_keys}
df_dict_cdp["current_tables"] = {key: df_dict_cdp["current_tables"][key] for key in common_keys}

#Keep common keys
common_keys = set(df_dict_cda["reference_tables"].keys()) & set(df_dict_cda["current_tables"].keys())
df_dict_cda["reference_tables"] = {key: df_dict_cda["reference_tables"][key] for key in common_keys}
df_dict_cda["current_tables"] = {key: df_dict_cda["current_tables"][key] for key in common_keys}

#Keep common keys
common_keys = set(df_dict_cdp["reference_tables"].keys()) & set(df_dict_cdp["current_tables"].keys())
df_dict_cdp["reference_tables"] = {key: df_dict_cdp["reference_tables"][key] for key in common_keys}
df_dict_cdp["current_tables"] = {key: df_dict_cdp["current_tables"][key] for key in common_keys}

# COMMAND ----------

print('df_dict_cda["reference_tables"]',df_dict_cda["reference_tables"].keys())
print('df_dict_cda["current_tables"]',df_dict_cda["current_tables"].keys())
print('df_dict_cdp["reference_tables"]',df_dict_cdp["reference_tables"].keys())
print('df_dict_cdp["current_tables"]',df_dict_cdp["current_tables"].keys())

# COMMAND ----------

def cd_in_memory(cda_reference_df,cdp_reference_df,cda_current_df,cdp_current_df,listed_concept_monitors):
    concept_drift_metrics_list = monitor_concept_drift_in_memory(
    data_type="tabular",
    model_task='',
    base_y_actuals=cda_reference_df,
    base_y_predicted=cdp_reference_df,
    current_y_actuals=cda_current_df,
    current_y_predicted=cdp_current_df,
    monitoring_algos=listed_concept_monitors,
    monitoring_algo_thresholds= None,)
    for i,cd_metric in enumerate(concept_drift_metrics_list):
        concept_drift_metrics_list[i]["is_drift"] = cd_metric["is_drift"][0]
        concept_drift_metrics_list[i]["stat_val"] = cd_metric["stat_val"][0]
        concept_drift_metrics_list[i]["p_val"] = cd_metric["p_val"][0]
    return concept_drift_metrics_list

# COMMAND ----------

def create_cd_metric_json_list(concept_drift_metrics_list,segment):
    concept_drift_metric_json_list = []
    for i, metric in enumerate(concept_drift_metrics_list):
        concept_drift_metric_json = monitoring_metrics_template_json.copy()
        concept_drift_metric_json["granularity"] = segment
        concept_drift_metric_json["monitoring_type"] = "concept_drift"
        concept_drift_metric_json["monitoring_sub_type"] = concept_drift_conf[
            "modelling_task_type"
        ]
        concept_drift_metric_json["monitoring_algo"] = concept_drift_conf[
            "monitoring_algo"
        ][i]
        metric.pop("monitor_algo", None)
        concept_drift_metric_json["data"] = round_metrics(metric)
        print(f"CONCEPT_DRIFT ========================> {concept_drift_metric_json}")
        utils.log("CONCEPT_DRIFT Calculation completed", message_run)
        concept_drift_metric_json_list.append(concept_drift_metric_json)
    return concept_drift_metric_json_list

# COMMAND ----------

def cda_in_memory_segmented(df_dict_cda,df_dict_cdp,concept_drift_conf):
    FINAL_CDA_LIST = []
    concept_monitor_algo_to_setup, listed_concept_monitors = check_for_monitor_setup(concept_drift_conf, target_type=None)
    for k in df_dict_cda['reference_tables'].keys():
        data_type =  data_type_detection(feature_drift_conf,target_drift_conf)
        cda_reference_df = df_dict_cda['reference_tables'][k][actual_target]
        cdp_reference_df = df_dict_cdp["reference_tables"][k][predicted_target]
        cda_current_df = df_dict_cda['current_tables'][k][actual_target]
        cdp_current_df = df_dict_cdp["current_tables"][k][predicted_target]
        concept_drift_metrics_list = cd_in_memory(cda_reference_df,cdp_reference_df,cda_current_df,cdp_current_df,listed_concept_monitors)
        print(concept_drift_metrics_list)
        concept_drift_metric_json_list = create_cd_metric_json_list(concept_drift_metrics_list,str(k))
        FINAL_CDA_LIST.extend(concept_drift_metric_json_list)
    return FINAL_CDA_LIST

# COMMAND ----------

concept_drift_metric_json_list = cda_in_memory_segmented(df_dict_cda,df_dict_cdp,concept_drift_conf)

# COMMAND ----------

concept_drift_metric_json_list

# COMMAND ----------

len(concept_drift_metric_json_list)

# COMMAND ----------

# MAGIC %md
# MAGIC MODIFIED ABOVE

# COMMAND ----------

# cda_reference_df

# COMMAND ----------

# cdp_reference_df

# COMMAND ----------

# cda_current_df

# COMMAND ----------

# cdp_current_df

# COMMAND ----------

# concept_drift_conf

# COMMAND ----------

# MAGIC %md
# MAGIC #### Check if Artifacts have been setup for Concept Drift Config

# COMMAND ----------

# concept_monitor_algo_to_setup, listed_concept_monitors = check_for_monitor_setup(
#     drift_conf=concept_drift_conf, target_type=None
# )

# COMMAND ----------

# MAGIC %md
# MAGIC #### Setup Concept Drift Monitoring Artifacts if needed

# COMMAND ----------

# setup_monitor(
#     monitor_algo_to_setup=concept_monitor_algo_to_setup,
#     drift_conf=concept_drift_conf,
#     data_type="tabular",
#     monitoring_artifact_path=monitoring_artifact_path,
#     base_y_actuals=cda_reference_df,
#     base_y_predicted=cdp_reference_df,
# )

# COMMAND ----------

# MAGIC %md
# MAGIC #### Monitor Concept Drift for Current DF

# COMMAND ----------

# concept_drift_metrics_list = monitor_current_data(
#     drift_conf=concept_drift_conf,
#     data_type="tabular",
#     monitoring_artifact_path=monitoring_artifact_path,
#     monitoring_algos=listed_concept_monitors,
#     current_y_actuals=cda_current_df,
#     current_y_predicted=cdp_current_df,
# )

# COMMAND ----------

# concept_drift_metrics_list

# COMMAND ----------

# MAGIC %md
# MAGIC #### Calculate Concept Drift Metrics JSON List

# COMMAND ----------

# concept_drift_metric_json_list = []
# for i, metric in enumerate(concept_drift_metrics_list):
#     concept_drift_metric_json = monitoring_metrics_template_json.copy()
#     concept_drift_metric_json["monitoring_type"] = "concept_drift"
#     concept_drift_metric_json["monitoring_sub_type"] = concept_drift_conf[
#         "modelling_task_type"
#     ]
#     concept_drift_metric_json["monitoring_algo"] = concept_drift_conf[
#         "monitoring_algo"
#     ][i]
#     metric.pop("monitor_algo", None)
#     concept_drift_metric_json["data"] = round_metrics(metric)
#     print(f"CONCEPT_DRIFT ========================> {concept_drift_metric_json}")
#     utils.log("CONCEPT_DRIFT Calculation completed", message_run)
#     concept_drift_metric_json_list.append(concept_drift_metric_json)

# COMMAND ----------

# MAGIC %md
# MAGIC ### PERFORMANCE DRIFT

# COMMAND ----------

# MAGIC %md
# MAGIC #### Performance Drift Inputs
# MAGIC

# COMMAND ----------

# MAGIC %md
# MAGIC MODIFIED BELOW

# COMMAND ----------

df_dict_pda = get_grouped_df_dict(granularity_levels,pda_reference_df,pda_current_df)
df_dict_pdp = get_grouped_df_dict(granularity_levels,pdp_reference_df,pdp_current_df)

# COMMAND ----------

 #Keep common keys
common_keys = set(df_dict_pda["reference_tables"].keys()) & set(df_dict_pdp["reference_tables"].keys())
df_dict_pda["reference_tables"] = {key: df_dict_pda["reference_tables"][key] for key in common_keys}
df_dict_pdp["reference_tables"] = {key: df_dict_pdp["reference_tables"][key] for key in common_keys}

#Keep common keys
common_keys = set(df_dict_pda["current_tables"].keys()) & set(df_dict_pdp["current_tables"].keys())
df_dict_pda["current_tables"] = {key: df_dict_pda["current_tables"][key] for key in common_keys}
df_dict_pdp["current_tables"] = {key: df_dict_pdp["current_tables"][key] for key in common_keys}

#Keep common keys
common_keys = set(df_dict_pda["reference_tables"].keys()) & set(df_dict_pda["current_tables"].keys())
df_dict_pda["reference_tables"] = {key: df_dict_pda["reference_tables"][key] for key in common_keys}
df_dict_pda["current_tables"] = {key: df_dict_pda["current_tables"][key] for key in common_keys}

#Keep common keys
common_keys = set(df_dict_pdp["reference_tables"].keys()) & set(df_dict_pdp["current_tables"].keys())
df_dict_pdp["reference_tables"] = {key: df_dict_pdp["reference_tables"][key] for key in common_keys}
df_dict_pdp["current_tables"] = {key: df_dict_pdp["current_tables"][key] for key in common_keys}

# COMMAND ----------

print('df_dict_pda["reference_tables"]',df_dict_pda["reference_tables"].keys())
print('df_dict_pda["current_tables"]',df_dict_pda["current_tables"].keys())
print('df_dict_pdp["reference_tables"]',df_dict_pdp["reference_tables"].keys())
print('df_dict_pdp["current_tables"]',df_dict_pdp["current_tables"].keys())

# COMMAND ----------

def create_pd_metric_json_list(performance_drift_metrics_list,segment):
    performance_drift_metric_json_list = []
    for i, metric in enumerate(performance_drift_metrics_list):
        performance_drift_metric_json = monitoring_metrics_template_json.copy()
        performance_drift_metric_json["granularity"] = segment
        performance_drift_metric_json["monitoring_type"] = "performance_drift"
        performance_drift_metric_json["monitoring_sub_type"] = performance_drift_conf[
            "modelling_task_type"
        ]
        performance_drift_metric_json["monitoring_algo"] = {
            "modelling_task_name": performance_drift_conf["modelling_task_type"],
            "algo_name": "performance_metrics",
        }
        performance_drift_metric_json["data"] = round_metrics(metric)
        print(
            f"PERFORMANCE_DRIFT ========================> {performance_drift_metric_json}"
        )
        utils.log("PERFORMANCE_DRIFT Calculation completed", message_run)
        performance_drift_metric_json_list.append(performance_drift_metric_json)
    return performance_drift_metric_json_list

# COMMAND ----------

def pd_in_memory(pda_reference_df,pdp_reference_df,pda_current_df,pdp_current_df,listed_performance_monitors,performance_drift_conf):
    performance_drift_metrics_list = monitor_performance_drift_in_memory(
    data_type="tabular",
    model_task=performance_drift_conf["modelling_task_type"].lower(),
    base_y_actuals=pda_reference_df,
    base_y_predicted=pdp_reference_df,
    monitoring_metrics=listed_performance_monitors,
    current_y_actuals=pda_current_df,
    current_y_predicted=pdp_current_df,
    monitoring_metrics_thresholds= None)

    return performance_drift_metrics_list

# COMMAND ----------

def pda_in_memory_segmented(df_dict_pda,df_dict_pdp,performance_drift_conf):
    FINAL_PDA_LIST = []
    performance_monitor_algo_to_setup, listed_performance_monitors = (
    check_for_monitor_setup(drift_conf=performance_drift_conf, target_type=None))
    for k in df_dict_pda['reference_tables'].keys():
        data_type =  data_type_detection(feature_drift_conf,target_drift_conf)
        pda_reference_df = df_dict_pda['reference_tables'][k][actual_target]
        pdp_reference_df = df_dict_pdp["reference_tables"][k][predicted_target]
        pda_current_df = df_dict_pda['current_tables'][k][actual_target]
        pdp_current_df = df_dict_pdp["current_tables"][k][predicted_target]
        performance_drift_metrics_list = pd_in_memory(pda_reference_df,pdp_reference_df,pda_current_df,
        pdp_current_df,listed_performance_monitors,performance_drift_conf)
        performance_drift_metric_json_list = create_pd_metric_json_list([performance_drift_metrics_list],str(k))
        FINAL_PDA_LIST.extend(performance_drift_metric_json_list)
    return FINAL_PDA_LIST

# COMMAND ----------

performance_drift_metric_json_list = pda_in_memory_segmented(df_dict_pda,df_dict_pdp,performance_drift_conf)

# COMMAND ----------

len(performance_drift_metric_json_list)

# COMMAND ----------

# MAGIC %md
# MAGIC MODIFIED ABOVE

# COMMAND ----------

# pda_reference_df

# COMMAND ----------

# pdp_reference_df

# COMMAND ----------

# pda_current_df

# COMMAND ----------

# pdp_reference_df

# COMMAND ----------

# performance_drift_conf

# COMMAND ----------

# MAGIC %md
# MAGIC #### Check if Artifacts have been setup for Performance Drift Config

# COMMAND ----------

# performance_monitor_algo_to_setup, listed_performance_monitors = (
#     check_for_monitor_setup(drift_conf=performance_drift_conf, target_type=None)
# )

# COMMAND ----------

# MAGIC %md
# MAGIC #### Setup Performance Drift Monitoring Artifacts if needed

# COMMAND ----------

# setup_monitor(
#     monitor_algo_to_setup=performance_monitor_algo_to_setup,
#     drift_conf=performance_drift_conf,
#     data_type="tabular",
#     monitoring_artifact_path=monitoring_artifact_path,
#     base_y_actuals=pda_reference_df,
#     base_y_predicted=pdp_reference_df,
# )

# COMMAND ----------

# MAGIC %md
# MAGIC #### Monitor Performance Drift for Current DF

# COMMAND ----------

# performance_drift_metrics_list = monitor_current_data(
#     drift_conf=performance_drift_conf,
#     data_type="tabular",
#     monitoring_artifact_path=monitoring_artifact_path,
#     monitoring_algos=listed_performance_monitors,
#     current_y_actuals=pda_current_df,
#     current_y_predicted=pdp_current_df,
# )

# COMMAND ----------

# performance_drift_metrics_list

# COMMAND ----------

# MAGIC %md
# MAGIC #### Calculate Performance Drift Metrics JSON List

# COMMAND ----------

# performance_drift_metric_json_list = []
# for i, metric in enumerate(performance_drift_metrics_list):
#     performance_drift_metric_json = monitoring_metrics_template_json.copy()
#     performance_drift_metric_json["monitoring_type"] = "performance_drift"
#     performance_drift_metric_json["monitoring_sub_type"] = performance_drift_conf[
#         "modelling_task_type"
#     ]
#     performance_drift_metric_json["monitoring_algo"] = {
#         "modelling_task_name": performance_drift_conf["modelling_task_type"],
#         "algo_name": "performance_metrics",
#     }
#     performance_drift_metric_json["data"] = round_metrics(metric)
#     print(
#         f"PERFORMANCE_DRIFT ========================> {performance_drift_metric_json}"
#     )
#     utils.log("PERFORMANCE_DRIFT Calculation completed", message_run)
#     performance_drift_metric_json_list.append(performance_drift_metric_json)

# COMMAND ----------

# MAGIC %md
# MAGIC ### Defining metrics JSON

# COMMAND ----------

drifted_monitor_types = []
aggregated_is_drift = False

for metric_json in (
    feature_drift_metric_json_list
    + target_drift_metric_json_list
    + concept_drift_metric_json_list
    + performance_drift_metric_json_list
):
    # Checking if there is a drift and adding the subtype in aggregated drift types if the monitor type has drifted
    if isinstance(metric_json["data"]["is_drift"], list) and (
        1 in metric_json["data"]["is_drift"]
    ):
        aggregated_is_drift = True
        drifted_monitor_types.append(metric_json["monitoring_type"])
    else:
        if metric_json["data"]["is_drift"] == 1:
            aggregated_is_drift = True
            drifted_monitor_types.append(metric_json["monitoring_type"])

    # Encoding JSON
    metric_json = json_encode_nans(metric_json)

    # Determination of start and end markers
    if metric_json.get("monitoring_type") == "feature_drift":
        metric_df = spark.createDataFrame(current_df)
        start_marker = metric_df.select(F.min("id")).collect()[0][0]
        end_marker = metric_df.select(F.max("id")).collect()[0][0]
        table_name = "dpd_table"

    elif metric_json.get("monitoring_type") == "target_drift":
        if metric_json.get("monitoring_sub_type") == "actual":
            metric_df = spark.createDataFrame(tda_current_df)
            start_marker = metric_df.select(F.min("id")).collect()[0][0]
            end_marker = metric_df.select(F.max("id")).collect()[0][0]
            table_name = "y_actual_Table"
        else:
            metric_df = spark.createDataFrame(tdp_current_df)
            start_marker = metric_df.select(F.min("id")).collect()[0][0]
            end_marker = metric_df.select(F.max("id")).collect()[0][0]
            table_name = "inference_table"

    elif metric_json.get("monitoring_type") in ["concept_drift", "performance_drift"]:
        if metric_json.get("monitoring_type") == "concept_drift":
            metric_df = spark.createDataFrame(curr_df_w_yact_ypred_conc)
        else:
            metric_df = spark.createDataFrame(curr_df_w_yact_ypred_perf)
        start_marker = metric_df.select(F.min("id")).collect()[0][0]
        end_marker = metric_df.select(F.max("id")).collect()[0][0]
        table_name = "y_actual_Table"

    # Creating batch markers dict
    batch_markers = {
        "table_name": table_name,
        "start_marker": start_marker,
        "end_marker": end_marker,
    }
    metric_json["batch_markers"] = batch_markers
    metric_json["job_id"] = job_id
    metric_json["run_id"] = run_id

# COMMAND ----------

print(f"feature_drift_metric_json_list =======> {feature_drift_metric_json_list}")

# COMMAND ----------

print(f"target_drift_metric_json_list =======> {target_drift_metric_json_list}")

# COMMAND ----------

print(f"concept_drift_metric_json_list ======> {concept_drift_metric_json_list}")

# COMMAND ----------

print(f"performance_drift_metric_json_list ===> {performance_drift_metric_json_list}")

# COMMAND ----------

try:
    metrics_map = {
        "feature_drift": feature_drift_metric_json_list,
        "target_drift": target_drift_metric_json_list,
        "concept_drift": concept_drift_metric_json_list,
        "performance_drift": performance_drift_metric_json_list,
    }

    for metric_kind, metric_json in metrics_map.items():
        utils.log(metric_kind, message_run)
        if len(metric_json) > 0:
            write_metrics_to_delta(metric_json, metric_kind)
except Exception as e:
    traceback.print_exc()
    throw_exception(e)

# COMMAND ----------

if datalake_env == "delta":
    register_delta_as_hive(
        db_name=f"mlcore_observability_{deployment_env}",
        table_name=aggregated_dd_table_name,
        dbfs_path=aggregated_dd_drift_path,
        spark=spark,
    )
    spark.sql(f"OPTIMIZE delta.`{aggregated_dd_drift_path}` ZORDER BY (model_artifact_id)")

    register_delta_as_hive(
        db_name=f"mlcore_observability_{deployment_env}",
        table_name=aggregated_pd_table_name,
        dbfs_path=aggregated_performance_drift_path,
        spark=spark,
    )
    spark.sql(f"OPTIMIZE delta.`{aggregated_performance_drift_path}` ZORDER BY (model_artifact_id)")

# COMMAND ----------

# Pushing tables in Mongo
try:
    table_types_subtypes = {
        "Monitoring_Output": ["Data_Model_Segmented", "Performance_Segmented"],
        "Task_Log": "Inference_Batch",
    }

    for table_type, table_sub_type in table_types_subtypes.items():
        if isinstance(table_sub_type, list):
            for ind_table_sub_type in table_sub_type:
                if ind_table_sub_type.lower() == "data_model_segmented":
                    if (
                        metrics_map.get("feature_drift", []) == []
                        and metrics_map.get("target_drift", []) == []
                        and metrics_map.get("concept_drift", []) == []
                    ):
                        utils.log(
                            f"Since feature drift, concept drift and target drift is empty. Skip pushing table to MLCore",
                            message_run,
                        )
                        continue
                if (
                    ind_table_sub_type.lower() == "performance_segmented"
                    and metrics_map.get("performance_drift", []) == []
                ):
                    utils.log(
                        f"Since performance is empty. Skip pushing table to MLCore",
                        message_run,
                    )
                    continue

                # Checking if the table has already been created by an earlier run, if not, creating one
                is_table_present = table_exists(
                    table_type, ind_table_sub_type, job_id, version, project_id
                )

                # Calling Tables Add API
                if not is_table_present:
                    # Pushing table in Mongo
                    add_table_in_mongo(table_type, ind_table_sub_type)
        else:
            # Checking if the table has already been created by an earlier run, if not, creating one
            is_table_present = table_exists(
                table_type, table_sub_type, job_id, version, project_id
            )

            # Calling Tables Add API
            if not is_table_present:
                # Pushing table in Mongo
                add_table_in_mongo(table_type, table_sub_type)

except Exception as e:
    throw_exception(e)

# COMMAND ----------

# Pushing the aggregated tables tables in Mongo
try:
    agg_table_types_subtypes = {
        "Aggregated_Monitor_Output": ["Data_Model_Segmented", "Performance_Segmented"],
    }

    for table_type, table_sub_type in agg_table_types_subtypes.items():
        if isinstance(table_sub_type, list):
            for ind_table_sub_type in table_sub_type:
                # Checking if the table has already been created by an earlier run, if not, creating one
                is_table_present = table_exists(
                    table_type, ind_table_sub_type, "", "", "", True
                )

                # Calling Tables Add API
                if not is_table_present:
                    # Pushing table in Mongo
                    tables_add_payload = {
                        "name": (
                            aggregated_pd_table_name
                            if ind_table_sub_type.lower() == "performance_segmented"
                            else aggregated_dd_table_name
                        ),
                        "type": "Aggregated_Monitor_Output",
                        "sub_type": ind_table_sub_type,
                        "deltalake_path": "",
                        "created_by_id": str(created_by_id),
                        "created_by_name": created_by_name,
                        "updated_by_id": str(created_by_id),
                        "updated_by_name": created_by_name,
                        "job_id": "",
                        "project_id": "",
                        "version": "",
                        "task_id": "task0",
                        "created_run_id": "",
                        "status": "active",
                        "primary_keys": [],
                    }
                    if platform_datalake_env == "delta":
                        tables_add_payload["dbfs_path"] = (
                            aggregated_performance_drift_path
                            if ind_table_sub_type.lower() == "performance_segmented"
                            else aggregated_dd_drift_path
                        )
                        tables_add_payload["datalake_env"] = "delta"
                    else:
                        table_path = (
                            aggregated_performance_drift_path
                            if ind_table_sub_type.lower() == "performance_segmented"
                            else aggregated_dd_drift_path
                        )
                        tables_add_payload["db_path"] = (
                            f"{gcp_project_id}.{bq_database_name}.{table_path}"
                        )
                        tables_add_payload["datalake_env"] = platform_datalake_env

                    tables_add(tables_add_payload)

except Exception as e:
    traceback.print_exc()
    throw_exception(e)

# COMMAND ----------

# MAGIC %md
# MAGIC ##Declare job as successful

# COMMAND ----------

try:
    if len(feature_drift_metric_json_list) > 0:
        update_task_log_as_per_inference_table(
            monitoring_subtype="feature_drift",
            required_entry=current_df_inference_entry,
        )
    if len(actual_target_drift_metric_json_list) > 0:
        update_task_log_as_per_inference_table(
            monitoring_subtype="target_y_actual",
            required_entry=tda_current_df_inference_entry,
        )
    if len(predicted_target_drift_metric_json_list) > 0:
        update_task_log_as_per_inference_table(
            monitoring_subtype="target_y_pred",
            required_entry=tdp_current_df_inference_entry,
        )
    if len(concept_drift_metric_json_list) > 0:
        update_task_log_as_per_inference_table(
            monitoring_subtype="concept_drift",
            required_entry=curr_df_w_yact_ypred_con_inference_entry,
        )
    if len(performance_drift_metric_json_list) > 0:
        update_task_log_as_per_inference_table(
            monitoring_subtype="performance_drift",
            required_entry=curr_df_w_yact_ypred_perf_inference_entry,
        )
except Exception as e:
    traceback.print_exc()
    throw_exception(e)

# COMMAND ----------

# MAGIC %md
# MAGIC MODIFIED BELOW.
# MAGIC
# MAGIC RUN SUBMIT FOR GENERATING HTML CHARTS

# COMMAND ----------

try:
    dbutils.notebook.run(
        "Monitor_Report_Run_Submit",
        timeout_seconds=0,
        arguments={
            "sdk_session_id": sdk_session_id,
            "dd_path": f"dbfs:/user/hive/warehouse/{sdk_session_id}.db/data_model_drift_{job_id}",
            "pd_path": f"dbfs:/user/hive/warehouse/{sdk_session_id}.db/performance_drift_{job_id}",
            "monitor_job_id": job_id,
            "monitor_run_id": run_id,
            "project_id": project_id,
            "version": version,
            "env": env,
            "tracking_base_url" : "https://mlcoredevv2pg21.azurewebsites.net/",
        },
    )
except Exception as e:
    print(f"Exception while triggering Monitor Report notebook: {e}")

# COMMAND ----------

# MAGIC %md
# MAGIC MODIFIED ABOVE

# COMMAND ----------

if any(
    feature_drift_metric_json_list
    + actual_target_drift_metric_json_list
    + predicted_target_drift_metric_json_list
    + concept_drift_metric_json_list
    + performance_drift_metric_json_list
):
    declare_job_as_successful(
        skipped_due_to_gt=null_gt_skipped_types,
        drifted_monitor_types=drifted_monitor_types,
        aggregated_is_drift=aggregated_is_drift,
    )
else:
    declare_job_as_successful(
        skipped_due_to_gt=null_gt_skipped_types,
        drifted_monitor_types=drifted_monitor_types,
        aggregated_is_drift=aggregated_is_drift,
        processed_records="no",
    )